# Universal Acoustic Perturbation Masking

## Loading the Data

Loading the dataset from github

In [ ]:
!git clone https://github.com/k3rnel-pan1c-ksd/input-sounds.git

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("input-sounds/cropped_base_pretraining")

CLASSES = list("abcdefghijklmnopqrstuvwxyz")
CLASS_TO_IDX = {key: i for i, key in enumerate(CLASSES)}

SAMPLE_RATE = 44100
N_SAMPLES = 11025
N_FFT = 1024
HOP_LENGTH = 225
N_MELS = 64

print("TensorFlow:", tf.__version__)

In [ ]:
waveforms = []
labels = []
filenames = []

for key in CLASSES:
    for i in range(1, 126):
        path = DATA_DIR / f"{key}_{i}.wav"

        if not path.exists():
            continue

        audio, sr = librosa.load(path, sr=None, mono=True)

        if sr != SAMPLE_RATE:
            audio = librosa.resample(
                audio, orig_sr=sr, target_sr=SAMPLE_RATE
            )

        if len(audio) < N_SAMPLES:
            audio = np.pad(audio, (0, N_SAMPLES - len(audio)))
        else:
            audio = audio[:N_SAMPLES]

        waveforms.append(audio.astype(np.float32))
        labels.append(CLASS_TO_IDX[key])
        filenames.append(str(path))

waveforms = np.stack(waveforms)
labels = np.array(labels, dtype=np.int32)
filenames = np.array(filenames)

print("Waveforms:", waveforms.shape)
print("Labels:", labels.shape)
print("Number of classes:", len(np.unique(labels)))

In [ ]:
indices = np.arange(len(labels))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=labels[temp_idx]
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Important limitation: These recordings may share the same keyboard, environment, and recording session. A random split measures within-session generalization, not necessarily performance on a new recording setup.

In [ ]:
def extract_features(audio):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        pad_mode="constant"
    )

    log_mel = librosa.power_to_db(mel, ref=np.max)

    return log_mel.astype(np.float32)


X = np.stack([
    extract_features(audio)
    for audio in waveforms
])

X = X[..., np.newaxis]

print("Features:", X.shape)

In [ ]:
plt.figure(figsize=(10, 4))

plt.imshow(
    X[0, :, :, 0],
    aspect="auto",
    origin="lower",
    cmap="magma"
)

plt.title(f"Keystroke: {CLASSES[labels[0]]}")
plt.xlabel("Time frame")
plt.ylabel("Mel frequency bin")
plt.colorbar(label="dB")
plt.show()

## Fitting the Baseline CNN

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_cnn():
    model = keras.Sequential([
        keras.Input(shape=(64, 50, 1)),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Flatten(),

        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(len(CLASSES), activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


model = build_cnn()
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    keras.callbacks.ModelCheckpoint(
        "keystroke_cnn_best.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    X[train_idx],
    labels[train_idx],

    validation_data=(
        X[val_idx],
        labels[val_idx]
    ),

    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Classification Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Cross-Entropy Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(
    X[test_idx],
    labels[test_idx],
    verbose=0
)

print(f"Test accuracy: {test_acc:.2%}")

predictions = model.predict(X[test_idx], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)

print(classification_report(
    labels[test_idx],
    predicted_labels,
    target_names=CLASSES,
    zero_division=0
))

In [ ]:
import json

model.save("keystroke_cnn.keras")

np.savez_compressed(
    "keystroke_dataset.npz",
    waveforms=waveforms,
    labels=labels,
    filenames=filenames,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx
)

config = {
    "classes": CLASSES,
    "sample_rate": SAMPLE_RATE,
    "n_samples": N_SAMPLES,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "n_mels": N_MELS,
    "feature_type": "librosa log-mel power spectrogram",
    "db_reference": "per-sample maximum",
    "input_shape": [64, 50, 1]
}

with open("preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved model, dataset, and preprocessing configuration.")

## Reloading model

In [ ]:

import json
import numpy as np
import tensorflow as tf
import librosa

# ------------------------------------
# 1. Load the saved CNN
# ------------------------------------

model = tf.keras.models.load_model(
    "keystroke_cnn.keras",
    compile=False
)

print("Model loaded successfully.")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

# ------------------------------------
# 2. Load the saved dataset
# ------------------------------------

with np.load("keystroke_dataset.npz", allow_pickle=False) as data:
    waveforms = data["waveforms"]
    labels = data["labels"]
    filenames = data["filenames"]

    train_idx = data["train_idx"]
    val_idx = data["val_idx"]
    test_idx = data["test_idx"]

print("\nDataset loaded successfully.")
print("Waveforms:", waveforms.shape)
print("Labels:", labels.shape)
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

# ------------------------------------
# 3. Load preprocessing configuration
# ------------------------------------

with open("preprocessing_config.json", "r") as f:
    config = json.load(f)

print("\nConfiguration loaded successfully.")
print(config)

classes = config["classes"]

# ------------------------------------
# 4. Recreate the original preprocessing
# ------------------------------------

def extract_features(audio):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=config["sample_rate"],
        n_fft=config["n_fft"],
        hop_length=config["hop_length"],
        n_mels=config["n_mels"]
    )

    log_mel = librosa.power_to_db(
        mel,
        ref=np.max
    )

    return log_mel.astype(np.float32)

# ------------------------------------
# 5. Validate file compatibility
# ------------------------------------

assert waveforms.ndim == 2
assert waveforms.shape[1] == config["n_samples"]
assert len(waveforms) == len(labels)
assert model.output_shape[-1] == len(classes)

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

assert len(train_idx) + len(val_idx) + len(test_idx) == len(labels)

print("\nAll compatibility checks passed.")

# ------------------------------------
# 6. Run inference on one test example
# ------------------------------------

idx = int(test_idx[0])

audio = waveforms[idx]
true_label = classes[int(labels[idx])]

features = extract_features(audio)
features = features[np.newaxis, ..., np.newaxis]

assert tuple(features.shape[1:]) == tuple(model.input_shape[1:])

predictions = model(features, training=False).numpy()[0]

predicted_idx = int(np.argmax(predictions))
predicted_label = classes[predicted_idx]
confidence = float(predictions[predicted_idx])

print("\nInference test:")
print("Original file:", filenames[idx])
print("True label:", true_label)
print("Predicted label:", predicted_label)
print(f"Confidence: {confidence:.2%}")

# ------------------------------------
# 7. Re-evaluate the entire test split
# ------------------------------------

X_test = np.stack([
    extract_features(waveforms[i])
    for i in test_idx
])[..., np.newaxis]

test_predictions = model.predict(
    X_test,
    batch_size=32,
    verbose=0
)

test_accuracy = np.mean(
    np.argmax(test_predictions, axis=1) == labels[test_idx]
)

print(f"\nReloaded model test accuracy: {test_accuracy:.2%}")


In [ ]:

import json
import numpy as np
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt

with open("preprocessing_config.json") as f:
    config = json.load(f)

model = tf.keras.models.load_model(
    "keystroke_cnn.keras",
    compile=False
)
model.trainable = False

with np.load("keystroke_dataset.npz", allow_pickle=False) as data:
    waveforms = data["waveforms"]
    labels = data["labels"]
    test_idx = data["test_idx"]

SR = config["sample_rate"]
N_FFT = config["n_fft"]
HOP = config["hop_length"]
N_MELS = config["n_mels"]

# Use librosa's filter-bank coefficients, but convert them to
# a constant TensorFlow tensor for differentiable matrix multiplication.
mel_filters = librosa.filters.mel(
    sr=SR,
    n_fft=N_FFT,
    n_mels=N_MELS
).astype(np.float32)

MEL_FILTERS = tf.constant(mel_filters.T)

# Explicit periodic Hann window, consistent with librosa's STFT.
def hann_window(frame_length, dtype):
    return tf.signal.hann_window(
        frame_length,
        periodic=True,
        dtype=dtype
    )


def tf_power_to_db(mel):
    amin = tf.constant(1e-10, dtype=tf.float32)

    log_spec = 10.0 * (
        tf.math.log(tf.maximum(mel, amin))
        / tf.math.log(tf.constant(10.0, tf.float32))
    )

    # librosa ref=np.max is computed from the
    # original power spectrogram.
    ref_value = tf.reduce_max(
        mel,
        axis=[1, 2],
        keepdims=True
    )

    log_ref = 10.0 * (
        tf.math.log(tf.maximum(ref_value, amin))
        / tf.math.log(tf.constant(10.0, tf.float32))
    )

    log_spec = log_spec - log_ref

    # librosa's default top_db=80
    max_db = tf.reduce_max(
        log_spec,
        axis=[1, 2],
        keepdims=True
    )

    return tf.maximum(log_spec, max_db - 80.0)

def tf_log_mel(audio):
    """
    audio: [batch, samples], float32
    returns: [batch, 64, 50, 1]
    """
    audio = tf.convert_to_tensor(audio, dtype=tf.float32)

    # Match librosa STFT center=True and pad_mode="reflect".
    pad = N_FFT // 2
    audio = tf.pad(
        audio,
        [[0, 0], [pad, pad]],
        mode="CONSTANT"
    )

    stft = tf.signal.stft(
        audio,
        frame_length=N_FFT,
        frame_step=HOP,
        fft_length=N_FFT,
        window_fn=hann_window,
        pad_end=False
    )

    power = tf.square(tf.abs(stft))

    # [batch, frames, frequencies]
    #     × [frequencies, mel bands]
    # -> [batch, frames, mel bands]
    mel = tf.matmul(power, MEL_FILTERS)

    # librosa.power_to_db:
    # 10 * log10(max(power, amin) / max(ref, amin))
    amin = tf.constant(1e-10, dtype=tf.float32)

    log_power = 10.0 * (
        tf.math.log(tf.maximum(mel, amin))
        / tf.math.log(tf.constant(10.0, tf.float32))
    )

    # Per-example reference: np.max
    reference = tf.reduce_max(
        log_power,
        axis=[1, 2],
        keepdims=True
    )

    log_mel = log_power - reference

    # Match librosa's top_db=80
    maximum = tf.reduce_max(
        log_mel,
        axis=[1, 2],
        keepdims=True
    )

    log_mel = tf_power_to_db(mel)

    # CNN expects [batch, mel bands, time frames, channels].
    log_mel = tf.transpose(log_mel, [0, 2, 1])
    log_mel = log_mel[..., tf.newaxis]

    return log_mel


In [ ]:

def librosa_log_mel(audio):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP,
        n_mels=N_MELS,
        pad_mode="constant"
    )

    return librosa.power_to_db(
        mel,
        ref=np.max
    ).astype(np.float32)


# Select a few held-out examples.
audio_batch = waveforms[test_idx[:8]]

librosa_features = np.stack([
    librosa_log_mel(x)
    for x in audio_batch
])

tf_features = tf_log_mel(audio_batch).numpy()[..., 0]

print("Librosa shape:", librosa_features.shape)
print("TensorFlow shape:", tf_features.shape)

difference = np.abs(
    librosa_features - tf_features
)

print("Mean absolute error:", difference.mean())
print("Maximum absolute error:", difference.max())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(
    librosa_features[0],
    aspect="auto",
    origin="lower",
    vmin=-80,
    vmax=0
)
axes[0].set_title("Librosa")

axes[1].imshow(
    tf_features[0],
    aspect="auto",
    origin="lower",
    vmin=-80,
    vmax=0
)
axes[1].set_title("TensorFlow")

axes[2].imshow(
    difference[0],
    aspect="auto",
    origin="lower"
)
axes[2].set_title("Absolute difference")

plt.tight_layout()
plt.show()


In [ ]:

librosa_input = librosa_features[..., np.newaxis]
tf_input = tf_log_mel(audio_batch)

pred_librosa = model(
    librosa_input,
    training=False
).numpy()

pred_tf = model(
    tf_input,
    training=False
).numpy()

print(
    "Predictions match:",
    np.array_equal(
        np.argmax(pred_librosa, axis=1),
        np.argmax(pred_tf, axis=1)
    )
)

print(
    "Maximum probability difference:",
    np.max(np.abs(pred_librosa - pred_tf))
)

for i in range(len(audio_batch)):
    print(
        f"Sample {i}: "
        f"librosa={np.argmax(pred_librosa[i])}, "
        f"tensorflow={np.argmax(pred_tf[i])}"
    )


In [ ]:

audio = tf.Variable(
    waveforms[test_idx[:1]],
    dtype=tf.float32
)

target = tf.constant(
    labels[test_idx[:1]],
    dtype=tf.int32
)

with tf.GradientTape() as tape:
    features = tf_log_mel(audio)

    probabilities = model(
        features,
        training=False
    )

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        target,
        probabilities
    )

    loss = tf.reduce_mean(loss)

gradient = tape.gradient(loss, audio)

print("Loss:", float(loss))
print("Gradient exists:", gradient is not None)

if gradient is not None:
    print("Gradient shape:", gradient.shape)
    print(
        "Gradient norm:",
        float(tf.norm(gradient))
    )
    print(
        "All gradients finite:",
        bool(tf.reduce_all(tf.math.is_finite(gradient)))
    )


In [ ]:

# Error statistics by sample
for i in range(len(audio_batch)):
    diff = difference[i]

    print(
        f"Sample {i}: "
        f"MAE={diff.mean():.6f}, "
        f"Max={diff.max():.6f}, "
        f"95th percentile={np.percentile(diff, 95):.6f}"
    )

# Locate the largest discrepancy
sample_idx, mel_idx, frame_idx = np.unravel_index(
    np.argmax(difference),
    difference.shape
)

print("\nLargest discrepancy:")
print("Sample:", sample_idx)
print("Mel bin:", mel_idx)
print("Time frame:", frame_idx)
print("Librosa:", librosa_features[sample_idx, mel_idx, frame_idx])
print("TensorFlow:", tf_features[sample_idx, mel_idx, frame_idx])

# How many bins differ substantially?
for threshold in [0.01, 0.1, 1.0, 5.0, 10.0]:
    percentage = np.mean(difference > threshold) * 100
    print(f"Error > {threshold} dB: {percentage:.3f}%")


## Adversary Attack

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

model.trainable = False

# Convert data into TensorFlow datasets
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices((
    waveforms[train_idx].astype(np.float32),
    labels[train_idx].astype(np.int32)
))

train_dataset = (
    train_dataset
    .shuffle(len(train_idx), seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = tf.data.Dataset.from_tensor_slices((
    waveforms[test_idx].astype(np.float32),
    labels[test_idx].astype(np.int32)
))

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
EPSILON = 0.01
STEP_SIZE = 0.001
EPOCHS = 30

N_SAMPLES = waveforms.shape[1]

delta = tf.Variable(
    tf.zeros((N_SAMPLES,), dtype=tf.float32),
    trainable=True
)

print("Perturbation shape:", delta.shape)

In [ ]:

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False
)

@tf.function
def uap_train_step(audio_batch, label_batch, delta, epsilon, step_size):
    with tf.GradientTape() as tape:
        # Apply the same perturbation to all waveforms
        perturbed = audio_batch + delta[tf.newaxis, :]

        # Keep audio within valid waveform amplitude bounds
        perturbed = tf.clip_by_value(
            perturbed, -1.0, 1.0
        )

        # Differentiable preprocessing
        features = tf_log_mel(perturbed)

        # Frozen classifier
        predictions = model(
            features,
            training=False
        )

        # Maximize classification loss
        loss = loss_fn(label_batch, predictions)

    gradient = tape.gradient(loss, delta)

    # Sign-gradient ascent
    delta.assign_add(
        step_size * tf.sign(gradient)
    )

    # Project back into the L-infinity constraint
    delta.assign(
        tf.clip_by_value(
            delta, -epsilon, epsilon
        )
    )

    return loss


In [ ]:
def train_uap(delta, epsilon=EPSILON, step_size=STEP_SIZE, verbose=True):
  loss_history = []

  for epoch in range(EPOCHS):
      epoch_losses = []

      for audio_batch, label_batch in train_dataset:
          loss = uap_train_step(
              audio_batch,
              label_batch,
              delta,
              epsilon,
              step_size
          )

          epoch_losses.append(float(loss))

      mean_loss = np.mean(epoch_losses)
      loss_history.append(mean_loss)

      if verbose:
          print(
              f"Epoch {epoch + 1:02d}/{EPOCHS} "
              f"- Loss: {mean_loss:.4f}"
          )

  return loss_history

loss_history = train_uap(delta)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Training cross-entropy loss")
plt.title("Universal Perturbation Optimization")
plt.grid(True)
plt.show()

In [ ]:
def evaluate_perturbation(dataset, perturbation=None):
    total_correct = 0
    total_loss = 0.0
    total_samples = 0

    for audio_batch, label_batch in dataset:
        if perturbation is not None:
            audio_batch = tf.clip_by_value(
                audio_batch + perturbation[tf.newaxis, :],
                -1.0,
                1.0
            )

        features = tf_log_mel(audio_batch)

        predictions = model(
            features,
            training=False
        )

        batch_losses = tf.keras.losses.sparse_categorical_crossentropy(
            label_batch,
            predictions
        )

        predicted_labels = tf.argmax(
            predictions,
            axis=1,
            output_type=tf.int32
        )

        total_correct += int(
            tf.reduce_sum(
                tf.cast(
                    predicted_labels == label_batch,
                    tf.int32
                )
            )
        )

        total_loss += float(
            tf.reduce_sum(batch_losses)
        )

        total_samples += int(tf.shape(audio_batch)[0])

    return {
        "accuracy": total_correct / total_samples,
        "loss": total_loss / total_samples
    }


clean_results = evaluate_perturbation(test_dataset)
uap_results = evaluate_perturbation(test_dataset, delta)

print("Clean:", clean_results)
print("UAP:", uap_results)

In [ ]:
rng = np.random.default_rng(42)

random_noise = rng.uniform(
    -EPSILON,
    EPSILON,
    size=N_SAMPLES
).astype(np.float32)

random_noise = tf.constant(random_noise)

random_results = evaluate_perturbation(
    test_dataset,
    random_noise
)

print("\nComparison")
print(f"Clean accuracy:  {clean_results['accuracy']:.2%}")
print(f"Random accuracy: {random_results['accuracy']:.2%}")
print(f"UAP accuracy:    {uap_results['accuracy']:.2%}")

In [ ]:
delta_np = delta.numpy()

time_axis = np.arange(len(delta_np)) / SR

plt.figure(figsize=(12, 4))
plt.plot(time_axis, delta_np)
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title("Learned Universal Adversarial Perturbation")
plt.grid(True)
plt.show()

print("Maximum amplitude:", np.max(np.abs(delta_np)))
print("RMS amplitude:", np.sqrt(np.mean(delta_np ** 2)))

In [ ]:
from IPython.display import Audio, display

display(Audio(delta_np, rate=SR, normalize=False))

In [ ]:
sample_idx = test_idx[0]

original = waveforms[sample_idx]
perturbed = np.clip(original + delta_np, -1.0, 1.0)

print("Original:")
display(Audio(original, rate=SR, normalize=False))

print("Perturbed:")
display(Audio(perturbed, rate=SR, normalize=False))

In [ ]:
np.save("uap_epsilon_001.npy", delta.numpy())

# Reload it
loaded_delta = np.load("uap_epsilon_001.npy")

print("Shape:", loaded_delta.shape)
print("Maximum amplitude:", np.max(np.abs(loaded_delta)))
print("Matches original:", np.allclose(loaded_delta, delta.numpy()))

import soundfile as sf

sf.write(
    "uap_epsilon_001.wav",
    delta.numpy(),
    SR,
    subtype="FLOAT"
)

In [ ]:
from collections import Counter

def determine_common_classes(test_dataset, delta):
  all_predictions = []
  all_true_labels = []

  for audio_batch, label_batch in test_dataset:
      perturbed = tf.clip_by_value(
          audio_batch + delta[tf.newaxis, :],
          -1.0, 1.0
      )

      probabilities = model(
          tf_log_mel(perturbed),
          training=False
      )

      predictions = tf.argmax(
          probabilities, axis=1
      ).numpy()

      all_predictions.extend(predictions)
      all_true_labels.extend(label_batch.numpy())

  counts = Counter(all_predictions)

  print("Most common predicted classes:")
  for idx, count in counts.most_common(10):
      print(
          f"{config['classes'][idx]}: "
          f"{count} predictions"
      )

  print("Number of distinct predicted classes:", len(counts))

determine_common_classes(test_dataset, delta)

#### Examining the UAP and determining what the model learned

In [ ]:
rng = np.random.default_rng(123)

uap_rms = np.sqrt(np.mean(delta.numpy() ** 2))

random_waveform = rng.normal(
    size=delta.shape
).astype(np.float32)

random_waveform /= np.sqrt(
    np.mean(random_waveform ** 2)
)

random_waveform *= uap_rms

# Check its peak amplitude.
print("Random RMS:", np.sqrt(np.mean(random_waveform ** 2)))
print("Random peak:", np.max(np.abs(random_waveform)))

In [ ]:
uap = delta.numpy().copy()

def rms(x):
    return np.sqrt(np.mean(np.square(x)))

print("Original UAP:")
print("RMS:", rms(uap))
print("Peak:", np.max(np.abs(uap)))
print("Mean:", np.mean(uap))

rng = np.random.default_rng(42)

shuffled = rng.permutation(uap).astype(np.float32)

print("\nShuffled UAP:")
print("RMS:", rms(shuffled))
print("Peak:", np.max(np.abs(shuffled)))
print("Mean:", np.mean(shuffled))

assert np.isclose(rms(uap), rms(shuffled))
assert np.max(np.abs(uap)) == np.max(np.abs(shuffled))

original_results = evaluate_perturbation(
    test_dataset,
    tf.constant(uap)
)

shuffled_results = evaluate_perturbation(
    test_dataset,
    tf.constant(shuffled)
)

print(f"Original UAP: {original_results['accuracy']:.2%}")
print(f"Shuffled UAP: {shuffled_results['accuracy']:.2%}")

In [ ]:
rng = np.random.default_rng(42)

N_TRIALS = 30
shuffled_accuracies = []

for trial in range(N_TRIALS):
    shuffled = rng.permutation(uap).astype(np.float32)

    result = evaluate_perturbation(
        test_dataset,
        tf.constant(shuffled)
    )

    shuffled_accuracies.append(result["accuracy"])

shuffled_accuracies = np.array(shuffled_accuracies)

print(f"Original UAP accuracy: {original_results['accuracy']:.2%}")
print(f"Shuffled mean accuracy: {shuffled_accuracies.mean():.2%}")
print(f"Shuffled std: {shuffled_accuracies.std():.2%}")
print(f"Shuffled min: {shuffled_accuracies.min():.2%}")
print(f"Shuffled max: {shuffled_accuracies.max():.2%}")

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    shuffled_accuracies * 100,
    bins=10,
    edgecolor="black",
    alpha=0.7
)

plt.axvline(
    original_results["accuracy"] * 100,
    color="red",
    linestyle="--",
    label="Original UAP"
)

plt.xlabel("Classifier Accuracy (%)")
plt.ylabel("Number of Shuffles")
plt.title("Original UAP vs. Shuffled Perturbations")
plt.legend()
plt.show()

In [ ]:
b_idx = config["classes"].index("b")

b_probabilities = []
true_labels = []
predicted_labels = []

for audio_batch, label_batch in test_dataset:
    perturbed = tf.clip_by_value(
        audio_batch + delta[tf.newaxis, :],
        -1.0,
        1.0
    )

    probabilities = model(
        tf_log_mel(perturbed),
        training=False
    ).numpy()

    b_probabilities.extend(probabilities[:, b_idx])
    predicted_labels.extend(np.argmax(probabilities, axis=1))
    true_labels.extend(label_batch.numpy())

b_probabilities = np.array(b_probabilities)
true_labels = np.array(true_labels)
predicted_labels = np.array(predicted_labels)

print("Mean P(b):", b_probabilities.mean())
print("Median P(b):", np.median(b_probabilities))
print("Minimum P(b):", b_probabilities.min())
print("Maximum P(b):", b_probabilities.max())

print("P(b) > 0.90:", np.mean(b_probabilities > 0.90))
print("P(b) > 0.99:", np.mean(b_probabilities > 0.99))

It seems the UAP produced relies on the order and temporal component of the waveform, and very confidently makes the model predict the wrong key

In [ ]:
epsilons = [0, 0.0005, 0.001, 0.002, 0.003, 0.005, 0.0075, 0.01]
original_uap = delta.numpy()
b_idx = config["classes"].index("b")

for epsilon in epsilons:
  ep_result = evaluate_perturbation(test_dataset, original_uap * (epsilon / EPSILON))
  scaled = original_uap * (epsilon / EPSILON)

  predictions = []

  for audio_batch, _ in test_dataset:
      perturbed = tf.clip_by_value(
          audio_batch + scaled[tf.newaxis, :],
          -1.0, 1.0
      )

      probs = model(tf_log_mel(perturbed), training=False)

      predictions.extend(tf.argmax(probs, axis=1).numpy())

  predictions = np.array(predictions)

  fraction_b = np.mean(predictions == b_idx)
  print(f"Scaled original UAP (epsilon={epsilon}): {ep_result['accuracy']:.4%};  ----  {fraction_b:.2%} predicted as 'b'")

In [ ]:
epsilons = [0.003]
trained_uaps = {}

tf.keras.utils.set_random_seed(696)
np.random.seed(696)
rng = np.random.default_rng(696)

for epsilon in epsilons:
  new_delta = tf.Variable(
      tf.zeros((N_SAMPLES,), dtype=tf.float32),
      trainable=True
  )

  print(f"--- Training Using Epsilon: {epsilon} ---")
  step_size = 0.1 * epsilon
  train_uap(new_delta, epsilon=epsilon, step_size=step_size, verbose=False)

  print("UAP results:", evaluate_perturbation(test_dataset, new_delta))
  determine_common_classes(test_dataset, new_delta.numpy())

  trained_uaps[epsilon] = new_delta.numpy().copy()

In [ ]:
for eps, perturbation in trained_uaps.items():
    p = np.asarray(perturbation, dtype=np.float32)

    peak = np.max(np.abs(p))
    rms = np.sqrt(np.mean(p ** 2))

    print(
        f"epsilon={eps:.4f} | "
        f"peak={peak:.6f} | "
        f"RMS={rms:.6f}"
    )

    assert peak <= eps + 1e-7

#### investigating the frequency distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

SR = 44100

def get_waveform(value):
    if isinstance(value, tf.Variable):
        return value.numpy().copy()
    return np.asarray(value, dtype=np.float32).copy()

uap = get_waveform(trained_uaps[0.003])

def rms(x):
    return np.sqrt(np.mean(np.square(x)))

print("Peak:", np.max(np.abs(uap)))
print("RMS:", rms(uap))
print("Duration:", len(uap) / SR)

frequencies, psd = welch(
    uap,
    fs=SR,
    nperseg=2048,
    noverlap=1024
)

plt.figure(figsize=(11, 4))
plt.plot(
    frequencies,
    10 * np.log10(psd + 1e-20)
)

plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (dB relative to 1 amplitude²/Hz)")
plt.title("Power Spectrum of Learned UAP (ε = 0.003)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
bands = [
    (0, 500),
    (500, 2000),
    (2000, 6000),
    (6000, 12000),
    (12000, SR / 2)
]

df = frequencies[1] - frequencies[0]
total_power = np.sum(psd) * df

for low, high in bands:
    mask = (
        (frequencies >= low) &
        (frequencies < high)
    )

    band_power = np.sum(psd[mask]) * df
    percentage = 100 * band_power / total_power

    print(
        f"{low:>5.0f}–{high:>7.0f} Hz: "
        f"{percentage:6.2f}%"
    )

#### Comparing the spectrograms

In [ ]:
def get_test_example(letter):
    class_idx = config["classes"].index(letter)

    for idx in test_idx:
        if labels[idx] == class_idx:
            return waveforms[idx].astype(np.float32)

    raise ValueError(f"No test example for {letter}")

audio_a = get_test_example("a")
audio_b = get_test_example("b")

masked_a = np.clip(audio_a + uap, -1.0, 1.0)

In [ ]:
import tensorflow as tf

signals = np.stack([
    audio_a,
    audio_b,
    masked_a,
    uap
]).astype(np.float32)

mel_features = tf_log_mel(signals).numpy()[..., 0]

titles = [
    "Clean 'a'",
    "Clean 'b'",
    "'a' + UAP",
    "UAP alone"
]

fig, axes = plt.subplots(
    1, 4,
    figsize=(16, 4),
    sharex=True,
    sharey=True
)

for ax, feature, title in zip(
    axes, mel_features, titles
):
    im = ax.imshow(
        feature,
        origin="lower",
        aspect="auto",
        vmin=-80,
        vmax=0,
        extent=[0, 0.25, 0, 64]
    )

    ax.set_title(title)
    ax.set_xlabel("Time (seconds)")

axes[0].set_ylabel("Mel-frequency bin")
fig.colorbar(
    im,
    ax=axes,
    label="dB relative to each signal's maximum",
    shrink=0.8
)

plt.show()

In [ ]:
clean_mel = mel_features[0]
masked_mel = mel_features[2]

difference = masked_mel - clean_mel

plt.figure(figsize=(9, 5))

plt.imshow(
    difference,
    origin="lower",
    aspect="auto",
    cmap="coolwarm",
    vmin=-40,
    vmax=40,
    extent=[0, 0.25, 0, 64]
)

plt.colorbar(label="Change in normalized log-mel value (dB)")
plt.xlabel("Time (seconds)")
plt.ylabel("Mel-frequency bin")
plt.title("Change in CNN Input After Adding UAP")
plt.show()

#### Comparing UAP To original audio energy

In [ ]:
test_audio = waveforms[test_idx].astype(np.float32)

# Account for any clipping after addition.
perturbed_audio = np.clip(
    test_audio + uap[None, :],
    -1.0, 1.0
)

applied_noise = perturbed_audio - test_audio

signal_rms = np.sqrt(
    np.mean(test_audio ** 2, axis=1)
)

noise_rms = np.sqrt(
    np.mean(applied_noise ** 2, axis=1)
)

snr_db = 20 * np.log10(
    (signal_rms + 1e-12) /
    (noise_rms + 1e-12)
)

print("Mean SNR:", snr_db.mean())
print("Median SNR:", np.median(snr_db))
print("Minimum SNR:", snr_db.min())
print("Maximum SNR:", snr_db.max())

print(
    "Fraction of samples clipped:",
    np.mean(
        np.abs(test_audio + uap[None, :]) > 1.0
    )
)

#### Frequency ablation

In [ ]:
def remove_frequency_band(signal, low_hz, high_hz, sr=44100):
    signal = np.asarray(signal, dtype=np.float32)

    spectrum = np.fft.rfft(signal)
    frequencies = np.fft.rfftfreq(len(signal), d=1/sr)

    mask = (frequencies >= low_hz) & (frequencies < high_hz)

    spectrum[mask] = 0

    filtered = np.fft.irfft(
        spectrum,
        n=len(signal)
    ).astype(np.float32)

    return filtered

In [ ]:
import numpy as np
import tensorflow as tf

uap = np.asarray(
    trained_uaps[0.003],
    dtype=np.float32
).copy()

epsilon = 0.003

def rms(x):
    return np.sqrt(np.mean(x ** 2))

bands = [
    (0, 500),
    (500, 2000),
    (2000, 6000),
    (6000, 12000),
    (12000, 22050)
]

original_rms = rms(uap)

original_results = evaluate_perturbation(
    test_dataset,
    tf.constant(uap)
)

print(f"Original accuracy: {original_results['accuracy']:.2%}")
print()

for low, high in bands:

    # Remove the selected frequency band.
    raw_filtered = remove_frequency_band(
        uap, low, high
    )

    raw_peak = np.max(np.abs(raw_filtered))

    # Apply the original peak constraint.
    gain = min(1.0, epsilon / max(raw_peak, 1e-12))
    filtered = raw_filtered * gain

    filtered_rms = rms(filtered)

    # Preserve all original frequencies but match RMS.
    scaled_original = uap * (
        filtered_rms / original_rms
    )

    filtered_results = evaluate_perturbation(
        test_dataset,
        tf.constant(filtered)
    )

    scaled_results = evaluate_perturbation(
        test_dataset,
        tf.constant(scaled_original)
    )

    print(f"Removed {low}-{high} Hz")
    print(f"  Raw filtered peak: {raw_peak:.6f}")
    print(f"  Applied gain:      {gain:.4f}")
    print(f"  Remaining RMS:     {filtered_rms:.6f}")
    print(f"  Filtered accuracy: {filtered_results['accuracy']:.2%}")
    print(f"  RMS-match accuracy:{scaled_results['accuracy']:.2%}")
    print()

#### Frequency masking

In [ ]:
SR = 44100
N_SAMPLES = 11025

def make_frequency_mask(
    low_hz,
    high_hz,
    exclude=False
):
    frequencies = np.fft.rfftfreq(
        N_SAMPLES,
        d=1 / SR
    )

    selected = (
        (frequencies >= low_hz) &
        (frequencies < high_hz)
    )

    if exclude:
        selected = ~selected

    return tf.constant(
        selected.astype(np.complex64)
    )

In [ ]:
def project_frequency_and_peak(
    waveform,
    frequency_mask,
    epsilon
):
    # Convert to frequency domain.
    spectrum = tf.signal.rfft(waveform)

    # Remove disallowed frequencies.
    spectrum = spectrum * frequency_mask

    # Return to time domain.
    filtered = tf.signal.irfft(
        spectrum,
        fft_length=[N_SAMPLES]
    )

    # Enforce maximum amplitude without
    # introducing new frequency components.
    peak = tf.reduce_max(tf.abs(filtered))

    scale = tf.minimum(
        1.0,
        epsilon / tf.maximum(peak, 1e-12)
    )

    return filtered * scale

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False
)

model.trainable = False

def frequency_project(waveform, frequency_mask):
    spectrum = tf.signal.rfft(waveform)

    filtered_spectrum = spectrum * frequency_mask

    return tf.signal.irfft(
        filtered_spectrum,
        fft_length=[N_SAMPLES]
    )

def project_bandlimited_box(
    candidate,
    frequency_mask,
    epsilon,
    iterations=40
):
    epsilon = tf.cast(epsilon, tf.float32)

    x = candidate
    p = tf.zeros_like(candidate)
    q = tf.zeros_like(candidate)

    # Dykstra's alternating projections.
    def body(i, x, p, q):

        # Projection onto amplitude constraint.
        z = x + p
        y = tf.clip_by_value(
            z, -epsilon, epsilon
        )
        p_new = z - y

        # Projection onto frequency constraint.
        z = y + q
        x_new = frequency_project(
            z, frequency_mask
        )
        q_new = z - x_new

        return i + 1, x_new, p_new, q_new

    _, x, _, _ = tf.while_loop(
        lambda i, x, p, q: i < iterations,
        body,
        (
            tf.constant(0),
            x,
            p,
            q
        )
    )

    # Preserve strict frequency support.
    # Final scaling guarantees peak feasibility
    # if the iterative solver has not fully converged.
    peak = tf.reduce_max(tf.abs(x))

    scale = tf.minimum(
        1.0,
        epsilon / tf.maximum(peak, 1e-12)
    )

    return x * scale

In [ ]:
@tf.function
def frequency_uap_train_step(
    audio_batch,
    label_batch,
    delta,
    epsilon,
    frequency_mask
):
    epsilon = tf.cast(epsilon, tf.float32)
    step_size = 0.1 * epsilon

    with tf.GradientTape() as tape:
        perturbed = tf.clip_by_value(
            audio_batch + delta[tf.newaxis, :],
            -1.0,
            1.0
        )

        features = tf_log_mel(perturbed)

        predictions = model(
            features,
            training=False
        )

        loss = loss_fn(
            label_batch,
            predictions
        )

    gradient = tape.gradient(loss, delta)

    # Restore original sign-gradient ascent.
    candidate = (
        delta + step_size * tf.sign(gradient)
    )

    # Project onto BOTH constraints.
    projected = project_bandlimited_box(
        candidate,
        frequency_mask,
        epsilon,
        iterations=40
    )

    delta.assign(projected)

    return loss

In [ ]:
def train_frequency_uap(
    epsilon,
    frequency_mask,
    epochs=30,
    verbose=True
):
    delta = tf.Variable(
        tf.zeros((N_SAMPLES,), dtype=tf.float32),
        trainable=True
    )

    history = []

    for epoch in range(epochs):
        losses = []

        for audio_batch, label_batch in train_dataset:
            loss = frequency_uap_train_step(
                audio_batch,
                label_batch,
                delta,
                epsilon,
                frequency_mask
            )

            losses.append(float(loss.numpy()))

        mean_loss = np.mean(losses)
        history.append(mean_loss)

        current = delta.numpy()

        current_peak = np.max(np.abs(current))
        current_rms = np.sqrt(np.mean(current ** 2))

        if verbose and (
            epoch == 0 or
            (epoch + 1) % 5 == 0
        ):
            print(
                f"Epoch {epoch + 1}: "
                f"Loss={mean_loss:.4f}, "
                f"Peak={current_peak:.6f}, "
                f"RMS={current_rms:.6f}"
            )

    return delta.numpy().copy(), history

In [ ]:
def verify_frequency_constraint(
    waveform,
    frequency_mask,
    epsilon
):
    waveform = np.asarray(
        waveform,
        dtype=np.float32
    )

    mask = np.abs(frequency_mask.numpy()) > 0

    spectrum = np.fft.rfft(waveform)
    power = np.abs(spectrum) ** 2

    total_power = np.sum(power)
    outside_power = np.sum(power[~mask])

    leakage = outside_power / max(total_power, 1e-30)

    peak = np.max(np.abs(waveform))
    rms = np.sqrt(np.mean(waveform ** 2))

    print(f"Peak: {peak:.8f}")
    print(f"RMS: {rms:.8f}")
    print(f"Power outside allowed band: {leakage:.3e}")

    assert peak <= epsilon + 1e-7
    assert leakage < 1e-6

In [ ]:
EPSILON = 0.003

experiments = {
    "full_spectrum": tf.ones(
        (N_SAMPLES // 2 + 1,),
        dtype=tf.complex64
    ),
    "500-2000_only": make_frequency_mask(
        500, 2000
    ),
    "2000-6000_only": make_frequency_mask(
        2000, 6000
    ),
    "500-6000_only": make_frequency_mask(
        500, 6000
    ),
    "6000-22050_only": make_frequency_mask(
        6000, 22050
    ),
    "500-6000_excluded": make_frequency_mask(
        500, 6000,
        exclude=True
    ),
    "500-2000_excluded": make_frequency_mask(
        500, 2000,
        exclude=True
    )
}

frequency_uaps = {}

for name, mask in experiments.items():
    print(f"\n--- {name} ---")

    uap, history = train_frequency_uap(
        epsilon=EPSILON,
        frequency_mask=mask,
        epochs=30,
        verbose=True
    )

    results = evaluate_perturbation(
        test_dataset,
        tf.constant(uap)
    )

    frequency_uaps[name] = {
        "waveform": uap,
        "history": history,
        "results": results
    }

    print("Accuracy:", results["accuracy"])
    print("Loss:", results["loss"])
    print("Peak:", np.max(np.abs(uap)))
    print("RMS:", np.sqrt(np.mean(uap ** 2)))

    verify_frequency_constraint(
        uap, mask, EPSILON
    )

In [ ]:
uap_only = frequency_uaps["500-2000_only"]["waveform"]
uap_excluded = frequency_uaps["500-2000_excluded"]["waveform"]

def rms(x):
    return np.sqrt(np.mean(np.asarray(x) ** 2))

target_rms = rms(uap_only)

scaled_excluded = uap_excluded * (
    target_rms / rms(uap_excluded)
)

print("Band-only RMS:", rms(uap_only))
print("Scaled excluded RMS:", rms(scaled_excluded))

for name, uap in [
    ("500-2000 only", uap_only),
    ("Excluded, RMS matched", scaled_excluded)
]:
    results = evaluate_perturbation(
        test_dataset,
        tf.constant(uap, dtype=tf.float32)
    )

    print(name, results)

In [ ]:
for name in [
    "full_spectrum",
    "500-2000_only",
    "2000-6000_only",
    "500-6000_only",
    "6000-22050_only",
    "500-6000_excluded",
    "500-2000_excluded"
]:
    print(f"\n--- {name} ---")

    uap = frequency_uaps[name]["waveform"]

    determine_common_classes(
        test_dataset,
        uap
    )

## Natural Masking

In [ ]:
import numpy as np
import librosa
import soundfile as sf

SR = 44100

audio_files = {
    "rain": "rain.wav",
    "fan": "fan.wav",
    "wind": "wind.wav"
}

reference_audio = {}

for name, path in audio_files.items():
    print(f"\n--- {name} ---")

    # Inspect original file metadata.
    info = sf.info(path)

    print(f"Original sample rate: {info.samplerate}")
    print(f"Original channels: {info.channels}")
    print(f"Original duration: {info.duration:.2f}")

    # Load, convert to mono, and resample to 44.1 kHz.
    audio, sr = librosa.load(
        path,
        sr=SR,
        mono=True
    )

    reference_audio[name] = audio

    print(f"Processed sample rate: {sr}")
    print(f"Processed shape: {audio.shape}")
    print(f"Processed duration: {len(audio)/sr:.2f}s")
    print(f"Peak amplitude: {np.max(np.abs(audio)):.6f}")
    print(f"RMS amplitude: {np.sqrt(np.mean(audio**2)):.6f}")

In [ ]:
from scipy.signal import welch
import matplotlib.pyplot as plt

reference_psds = {}

for name, audio in reference_audio.items():

    freqs, psd = welch(
        audio,
        fs=SR,
        window="hann",
        nperseg=2048,
        noverlap=1024,
        scaling="density"
    )

    # Normalize to compare spectral shapes,
    # independently of recording volume.
    psd = psd / (np.sum(psd) + 1e-12)

    reference_psds[name] = {
        "frequencies": freqs,
        "psd": psd
    }

plt.figure(figsize=(12, 5))

for name, data in reference_psds.items():
    plt.plot(
        data["frequencies"],
        10 * np.log10(data["psd"] + 1e-12),
        label=name
    )

plt.xlabel("Frequency (Hz)")
plt.ylabel("Normalized PSD (dB)")
plt.title("Environmental Audio Spectral Profiles")
plt.xlim(0, 22050)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
N_SAMPLES = 11025

uap_frequencies = np.fft.rfftfreq(
    N_SAMPLES,
    d=1 / SR
)

target_spectra = {}

for name, data in reference_psds.items():

    target = np.interp(
        uap_frequencies,
        data["frequencies"],
        data["psd"]
    )

    target /= np.sum(target) + 1e-12

    target_spectra[name] = target.astype(np.float32)

    print(name, target.shape)

In [ ]:
frequency_bands = [
    (0, 500),
    (500, 2000),
    (2000, 6000),
    (6000, 12000),
    (12000, 18000),
    (18000, 22050)
]

for name, data in reference_psds.items():
    freqs = data["frequencies"]
    psd = data["psd"]

    print(f"\n--- {name} ---")

    total_power = np.sum(psd)

    for low, high in frequency_bands:
        mask = (freqs >= low) & (freqs < high)

        band_power = np.sum(psd[mask])

        percentage = 100 * band_power / total_power

        print(
            f"{low:5d}-{high:5d} Hz: "
            f"{percentage:6.2f}%"
        )

In [ ]:
low_frequency_bands = [
    (0, 20),
    (20, 50),
    (50, 100),
    (100, 250),
    (250, 500),
    (500, 2000)
]

for name, data in reference_psds.items():
    freqs = data["frequencies"]
    psd = data["psd"]

    print(f"\n--- {name} ---")

    for low, high in low_frequency_bands:
        mask = (freqs >= low) & (freqs < high)

        percentage = 100 * (
            np.sum(psd[mask]) /
            np.sum(psd)
        )

        print(
            f"{low:4d}-{high:4d} Hz: "
            f"{percentage:.2f}%"
        )

In [ ]:
from scipy.signal import butter, sosfiltfilt, welch
import numpy as np

def highpass_audio(audio, cutoff=100, sr=44100):
    sos = butter(
        N=4,
        Wn=cutoff,
        btype="highpass",
        fs=sr,
        output="sos"
    )

    return sosfiltfilt(sos, audio).astype(np.float32)


filtered_reference_audio = {
    name: highpass_audio(audio, cutoff=100, sr=SR)
    for name, audio in reference_audio.items()
}

In [ ]:
filtered_reference_psds = {}

for name, audio in filtered_reference_audio.items():

    freqs, psd = welch(
        audio,
        fs=SR,
        window="hann",
        nperseg=2048,
        noverlap=1024,
        scaling="density"
    )

    psd = psd / (np.sum(psd) + 1e-12)

    filtered_reference_psds[name] = {
        "frequencies": freqs,
        "psd": psd
    }

for name, data in filtered_reference_psds.items():
    freqs = data["frequencies"]
    psd = data["psd"]

    print(f"\n--- {name} ---")

    for low, high in low_frequency_bands:
        mask = (freqs >= low) & (freqs < high)

        percentage = 100 * (
            np.sum(psd[mask]) /
            np.sum(psd)
        )

        print(
            f"{low:4d}-{high:4d} Hz: "
            f"{percentage:.2f}%"
        )

In [ ]:
import numpy as np
import tensorflow as tf
import librosa

from scipy.signal import welch

SR = 44100
N_SAMPLES = 11025
N_FFT = 1024
HOP_LENGTH = 225
N_MELS = 64
EPSILON = 0.003

# Same mel filterbank configuration as tf_log_mel.
mel_weights_np = librosa.filters.mel(
    sr=SR,
    n_fft=N_FFT,
    n_mels=N_MELS
).T.astype(np.float32)

mel_weights_tf = tf.constant(
    mel_weights_np,
    dtype=tf.float32
)


def make_reference_mel_distribution(audio):
    freqs, psd = welch(
        audio,
        fs=SR,
        window="hann",
        nperseg=N_FFT,
        noverlap=N_FFT - HOP_LENGTH,
        detrend=False,
        scaling="spectrum"
    )

    # Welch's one-sided PSD doubles interior bins.
    # Undo that to match raw rFFT power conventions.
    psd = psd.astype(np.float64)
    psd[1:-1] *= 0.5

    # Convert linear-frequency power to mel power.
    mel_power = psd @ mel_weights_np

    # Normalize to a probability distribution.
    mel_power = np.maximum(mel_power, 0.0)
    mel_power /= np.sum(mel_power) + 1e-20

    return mel_power.astype(np.float32)


reference_mel = {
    name: make_reference_mel_distribution(audio)
    for name, audio in filtered_reference_audio.items()
}

reference_mel_tf = {
    name: tf.constant(values, dtype=tf.float32)
    for name, values in reference_mel.items()
}

for name, spectrum in reference_mel.items():
    print(
        name,
        spectrum.shape,
        "sum:", spectrum.sum(),
        "peak band:", np.argmax(spectrum)
    )

In [ ]:
def uap_mel_distribution(delta):
    # STFT: [frames, frequency_bins]
    stft = tf.signal.stft(
        delta,
        frame_length=N_FFT,
        frame_step=HOP_LENGTH,
        fft_length=N_FFT,
        window_fn=tf.signal.hann_window,
        pad_end=False
    )

    # Linear power spectrum.
    power = tf.square(tf.abs(stft))

    # Average over time.
    mean_power = tf.reduce_mean(power, axis=0)

    # Project onto 64 mel bands.
    mel_power = tf.linalg.matvec(
        tf.transpose(mel_weights_tf),
        mean_power
    )

    # Normalize spectral shape.
    mel_power = mel_power + 1e-12
    mel_distribution = (
        mel_power / tf.reduce_sum(mel_power)
    )

    return mel_distribution


def environmental_loss(delta, target_mel):
    predicted = uap_mel_distribution(delta)

    # Squared Hellinger distance.
    distance = 0.5 * tf.reduce_sum(
        tf.square(
            tf.sqrt(predicted + 1e-12)
            - tf.sqrt(target_mel + 1e-12)
        )
    )

    return distance

In [ ]:
rng = np.random.default_rng(42)

test_delta = tf.Variable(
    rng.uniform(
        -EPSILON / 2,
        EPSILON / 2,
        size=N_SAMPLES
    ).astype(np.float32)
)

with tf.GradientTape() as tape:
    loss = environmental_loss(
        test_delta,
        reference_mel_tf["rain"]
    )

gradient = tape.gradient(loss, test_delta)

print("Environmental loss:", loss.numpy())
print("Gradient shape:", gradient.shape)
print("Gradient norm:", tf.norm(gradient).numpy())
print(
    "Gradient finite:",
    np.all(np.isfinite(gradient.numpy()))
)

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False
)

model.trainable = False


def calibrate_lambda(target_mel, seed=42):
    rng = np.random.default_rng(seed)

    # Use a nonzero probe because normalized spectral
    # power has zero gradient at an all-zero waveform.
    probe = tf.Variable(
        rng.uniform(
            -EPSILON / 2,
            EPSILON / 2,
            size=N_SAMPLES
        ).astype(np.float32)
    )

    audio_batch, label_batch = next(
        iter(train_dataset)
    )

    with tf.GradientTape(persistent=True) as tape:
        perturbed = tf.clip_by_value(
            audio_batch + probe[tf.newaxis, :],
            -1.0,
            1.0
        )

        predictions = model(
            tf_log_mel(perturbed),
            training=False
        )

        attack = loss_fn(
            label_batch,
            predictions
        )

        spectral = environmental_loss(
            probe,
            target_mel
        )

    attack_grad = tape.gradient(attack, probe)
    spectral_grad = tape.gradient(spectral, probe)

    del tape

    attack_norm = tf.norm(attack_grad).numpy()
    spectral_norm = tf.norm(spectral_grad).numpy()

    lambda_base = (
        attack_norm / max(spectral_norm, 1e-12)
    )

    print(f"Attack loss: {attack.numpy():.6f}")
    print(f"Spectral loss: {spectral.numpy():.6f}")
    print(f"Attack gradient norm: {attack_norm:.6f}")
    print(f"Spectral gradient norm: {spectral_norm:.6f}")
    print(f"Base lambda: {lambda_base:.6f}")

    return float(lambda_base)


lambda_bases = {}

for name, target in reference_mel_tf.items():
    print(f"\n--- {name} ---")

    lambda_bases[name] = calibrate_lambda(target)

In [ ]:
@tf.function
def environmental_uap_train_step(
    audio_batch,
    label_batch,
    delta,
    epsilon,
    target_mel,
    lambda_weight
):
    epsilon = tf.cast(epsilon, tf.float32)
    lambda_weight = tf.cast(
        lambda_weight, tf.float32
    )

    step_size = 0.1 * epsilon

    with tf.GradientTape() as tape:
        perturbed = tf.clip_by_value(
            audio_batch + delta[tf.newaxis, :],
            -1.0,
            1.0
        )

        predictions = model(
            tf_log_mel(perturbed),
            training=False
        )

        attack_loss = loss_fn(
            label_batch,
            predictions
        )

        spectral_loss = environmental_loss(
            delta,
            target_mel
        )

        # Maximize attack loss while minimizing
        # the spectral mismatch.
        combined_objective = (
            attack_loss
            - lambda_weight * spectral_loss
        )

    gradient = tape.gradient(
        combined_objective,
        delta
    )

    # Original sign-gradient ascent.
    delta.assign_add(
        step_size * tf.sign(gradient)
    )

    # Original elementwise L-infinity projection.
    delta.assign(
        tf.clip_by_value(
            delta,
            -epsilon,
            epsilon
        )
    )

    return attack_loss, spectral_loss

In [ ]:
def train_environmental_uap(
    target_mel,
    lambda_weight,
    epsilon=0.003,
    epochs=30,
    verbose=True
):
    delta = tf.Variable(
        tf.zeros(
            (N_SAMPLES,),
            dtype=tf.float32
        ),
        trainable=True
    )

    history = {
        "attack_loss": [],
        "spectral_loss": [],
        "rms": []
    }

    lambda_tensor = tf.constant(
        lambda_weight,
        dtype=tf.float32
    )

    for epoch in range(epochs):
        attack_losses = []
        spectral_losses = []

        for audio_batch, label_batch in train_dataset:
            attack, spectral = environmental_uap_train_step(
                audio_batch,
                label_batch,
                delta,
                epsilon,
                target_mel,
                lambda_tensor
            )

            attack_losses.append(float(attack.numpy()))
            spectral_losses.append(float(spectral.numpy()))

        current_delta = delta.numpy()

        mean_attack = np.mean(attack_losses)

        # Measure spectral distance AFTER the updates.
        current_spectral = float(
            environmental_loss(
                delta,
                target_mel
            ).numpy()
        )

        current_rms = np.sqrt(
            np.mean(current_delta ** 2)
        )

        history["attack_loss"].append(mean_attack)
        history["spectral_loss"].append(current_spectral)
        history["rms"].append(current_rms)

        if verbose and (
            epoch == 0 or
            (epoch + 1) % 5 == 0
        ):
            print(
                f"Epoch {epoch + 1:2d}/{epochs} | "
                f"Attack={mean_attack:.4f} | "
                f"Spectral={current_spectral:.4f} | "
                f"RMS={current_rms:.6f}"
            )

    return delta.numpy().copy(), history

todo: use validation set. not test set

In [ ]:
regularization_ratios = [0.0, 0.3, 1.0, 3.0]

rain_experiments = {}

for ratio in regularization_ratios:
    lambda_weight = (
        ratio * lambda_bases["rain"]
    )

    print(
        f"\n--- Rain | Ratio={ratio} | "
        f"Lambda={lambda_weight:.6f} ---"
    )

    uap, history = train_environmental_uap(
        target_mel=reference_mel_tf["rain"],
        lambda_weight=lambda_weight,
        epsilon=EPSILON,
        epochs=30
    )

    results = evaluate_perturbation(
        test_dataset,
        tf.constant(uap)
    )

    final_spectral_loss = float(
        environmental_loss(
            tf.constant(uap),
            reference_mel_tf["rain"]
        ).numpy()
    )

    rain_experiments[ratio] = {
        "waveform": uap,
        "history": history,
        "results": results,
        "lambda": lambda_weight,
        "spectral_loss": final_spectral_loss
    }

    print("Accuracy:", results["accuracy"])
    print("Attack loss:", results["loss"])
    print("Spectral loss:", final_spectral_loss)
    print("Peak:", np.max(np.abs(uap)))
    print("RMS:", np.sqrt(np.mean(uap ** 2)))

In [ ]:
import matplotlib.pyplot as plt

ratios = list(rain_experiments.keys())

accuracies = [
    rain_experiments[r]["results"]["accuracy"]
    for r in ratios
]

spectral_losses = [
    rain_experiments[r]["spectral_loss"]
    for r in ratios
]

fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(
    ratios,
    accuracies,
    marker="o",
    label="Classifier accuracy"
)

ax1.set_xlabel("Relative regularization strength")
ax1.set_ylabel("Classifier accuracy")
ax1.set_ylim(0, 1)

ax2 = ax1.twinx()

ax2.plot(
    ratios,
    spectral_losses,
    marker="s",
    linestyle="--",
    label="Spectral distance"
)

ax2.set_ylabel("Squared Hellinger distance")
ax2.set_ylim(0, 1)

plt.title("Rain: Attack Effectiveness vs Spectral Similarity")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    reference_mel["rain"],
    label="Reference rain",
    linewidth=2
)

for ratio in regularization_ratios:
    uap = rain_experiments[ratio]["waveform"]

    distribution = uap_mel_distribution(
        tf.constant(uap)
    ).numpy()

    plt.plot(
        distribution,
        label=f"UAP (ratio={ratio})",
        alpha=0.8
    )

plt.xlabel("Mel frequency band")
plt.ylabel("Normalized mel power")
plt.title("Rain Reference vs Learned UAP Spectra")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# original = waveforms[sample_idx]
# perturbed = np.clip(original + delta_np, -1.0, 1.0)

# print("Original:")
# display(Audio(original, rate=SR, normalize=False))

# print("Perturbed:")
# display(Audio(perturbed, rate=SR, normalize=False))

display(Audio(rain_experiments[0]['waveform'], rate=SR, normalize=False))

In [ ]:
def train_spectral_only(
    target_mel,
    epsilon=0.003,
    steps=500,
    learning_rate=2e-5,
    seed=42
):
    rng = np.random.default_rng(seed)

    initial = rng.uniform(
        -epsilon / 3,
        epsilon / 3,
        size=N_SAMPLES
    ).astype(np.float32)

    delta = tf.Variable(initial)

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    history = []

    for step in range(steps):
        with tf.GradientTape() as tape:
            loss = environmental_loss(
                delta,
                target_mel
            )

        gradient = tape.gradient(loss, delta)

        optimizer.apply_gradients([
            (gradient, delta)
        ])

        delta.assign(
            tf.clip_by_value(
                delta,
                -epsilon,
                epsilon
            )
        )

        current_loss = float(
            environmental_loss(delta, target_mel).numpy()
        )

        history.append(current_loss)

        if step == 0 or (step + 1) % 100 == 0:
            print(
                f"Step {step + 1}: "
                f"Spectral loss={current_loss:.6f}"
            )

    return delta.numpy().copy(), history


rain_spectral_uap, spectral_history = (
    train_spectral_only(
        reference_mel_tf["rain"]
    )
)

print(
    "Final spectral loss:",
    environmental_loss(
        tf.constant(rain_spectral_uap),
        reference_mel_tf["rain"]
    ).numpy()
)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(spectral_history)
plt.xlabel("Optimization step")
plt.ylabel("Squared Hellinger distance")
plt.title("Spectral-Only Optimization")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
results = evaluate_perturbation(
    test_dataset,
    tf.constant(rain_spectral_uap)
)

print("Spectral-only accuracy:", results["accuracy"])
print("Spectral-only attack loss:", results["loss"])
print(
    "RMS:",
    np.sqrt(np.mean(rain_spectral_uap ** 2))
)

In [ ]:
@tf.function
def environmental_uap_train_step_v2(
    audio_batch,
    label_batch,
    delta,
    epsilon,
    target_mel,
    beta
):
    epsilon = tf.cast(epsilon, tf.float32)
    beta = tf.cast(beta, tf.float32)

    step_size = 0.1 * epsilon

    with tf.GradientTape(persistent=True) as tape:
        perturbed = tf.clip_by_value(
            audio_batch + delta[tf.newaxis, :],
            -1.0,
            1.0
        )

        predictions = model(
            tf_log_mel(perturbed),
            training=False
        )

        attack_loss = loss_fn(
            label_batch,
            predictions
        )

        spectral_loss = environmental_loss(
            delta,
            target_mel
        )

    attack_grad = tape.gradient(
        attack_loss, delta
    )

    spectral_grad = tape.gradient(
        spectral_loss, delta
    )

    del tape

    # Normalize gradients independently.
    attack_grad = attack_grad / tf.maximum(
        tf.norm(attack_grad), 1e-12
    )

    spectral_grad = spectral_grad / tf.maximum(
        tf.norm(spectral_grad), 1e-12
    )

    # Increase attack loss, decrease spectral loss.
    combined_grad = (
        attack_grad - beta * spectral_grad
    )

    delta.assign_add(
        step_size * tf.sign(combined_grad)
    )

    delta.assign(
        tf.clip_by_value(
            delta,
            -epsilon,
            epsilon
        )
    )

    return attack_loss, spectral_loss


def train_environmental_uap_v2(
    target_mel,
    beta,
    epsilon=0.003,
    epochs=30,
    verbose=True
):
    delta = tf.Variable(
        tf.random.uniform(
            (N_SAMPLES,),
            dtype=tf.float32
        ),
        trainable=True
    )

    history = {
        "attack_loss": [],
        "spectral_loss": [],
        "rms": []
    }


    for epoch in range(epochs):
        attack_losses = []
        spectral_losses = []

        for audio_batch, label_batch in train_dataset:
            attack, spectral = environmental_uap_train_step(
                audio_batch,
                label_batch,
                delta,
                epsilon,
                target_mel,
                beta
            )

            attack_losses.append(float(attack.numpy()))
            spectral_losses.append(float(spectral.numpy()))

        current_delta = delta.numpy()

        mean_attack = np.mean(attack_losses)

        # Measure spectral distance AFTER the updates.
        current_spectral = float(
            environmental_loss(
                delta,
                target_mel
            ).numpy()
        )

        current_rms = np.sqrt(
            np.mean(current_delta ** 2)
        )

        history["attack_loss"].append(mean_attack)
        history["spectral_loss"].append(current_spectral)
        history["rms"].append(current_rms)

        if verbose and (
            epoch == 0 or
            (epoch + 1) % 5 == 0
        ):
            print(
                f"Epoch {epoch + 1:2d}/{epochs} | "
                f"Attack={mean_attack:.4f} | "
                f"Spectral={current_spectral:.4f} | "
                f"RMS={current_rms:.6f}"
            )

    return delta.numpy().copy(), history

In [ ]:
regularization_ratios = [0.0, 0.3, 1.0, 3.0]
beta = [0, 0.25, 0.5, 1, 2, 4]

rain_experiments = {}

for ratio in regularization_ratios:
  for b in beta:
    print(
        f"\n--- Rain | Ratio={ratio} | "
        f"beta={b} ---"
    )

    tf.random.set_seed(67)

    uap, history = train_environmental_uap_v2(
        target_mel=reference_mel_tf["rain"],
        beta=b,
        epsilon=EPSILON,
        epochs=30
    )

    results = evaluate_perturbation(
        test_dataset,
        tf.constant(uap)
    )

    final_spectral_loss = float(
        environmental_loss(
            tf.constant(uap),
            reference_mel_tf["rain"]
        ).numpy()
    )

    rain_experiments[ratio] = {
        "waveform": uap,
        "history": history,
        "results": results,
        "beta": b,
        "spectral_loss": final_spectral_loss
    }

    print("Accuracy:", results["accuracy"])
    print("Attack loss:", results["loss"])
    print("Spectral loss:", final_spectral_loss)
    print("Peak:", np.max(np.abs(uap)))
    print("RMS:", np.sqrt(np.mean(uap ** 2)))

In [ ]:
def saturation_fraction(delta, epsilon=0.003):
    delta = np.asarray(delta)
    return np.mean(
        np.abs(delta) >= 0.99 * epsilon
    )

print(
    "Saturated fraction:",
    saturation_fraction(
        rain_experiments[0.0]["waveform"]
    )
)

In [ ]:
def train_constrained_environmental_uap(
    initial_delta,
    target_mel,
    tau=0.02,
    epsilon=0.003,
    epochs=30,
    attack_lr=1e-5,
    spectral_lr=2e-5,
    max_repairs=20
):
    # Initialize from a waveform that already matches
    # the environmental spectrum.
    initial = np.asarray(
        initial_delta, dtype=np.float32
    ).copy()

    # Scale to utilize the available peak budget.
    peak = np.max(np.abs(initial))
    if peak > 0:
        initial *= epsilon / peak

    delta = tf.Variable(initial)

    attack_optimizer = tf.keras.optimizers.Adam(
        learning_rate=attack_lr
    )

    spectral_optimizer = tf.keras.optimizers.Adam(
        learning_rate=spectral_lr
    )

    history = {
        "attack_loss": [],
        "spectral_loss": [],
        "rms": [],
        "repair_steps": []
    }

    for epoch in range(epochs):
        attack_losses = []
        total_repairs = 0

        for audio_batch, label_batch in train_dataset:

            # Step 1: maximize attack loss.
            with tf.GradientTape() as tape:
                perturbed = tf.clip_by_value(
                    audio_batch + delta[tf.newaxis, :],
                    -1.0, 1.0
                )

                predictions = model(
                    tf_log_mel(perturbed),
                    training=False
                )

                attack_loss = loss_fn(
                    label_batch, predictions
                )

            attack_grad = tape.gradient(
                attack_loss, delta
            )

            attack_optimizer.apply_gradients([
                (-attack_grad, delta)
            ])

            delta.assign(
                tf.clip_by_value(
                    delta, -epsilon, epsilon
                )
            )

            # Step 2: repair spectral mismatch if needed.
            repairs = 0

            for _ in range(max_repairs):
                current_spectral = float(
                    environmental_loss(
                        delta, target_mel
                    ).numpy()
                )

                if current_spectral <= tau:
                    break

                with tf.GradientTape() as tape:
                    spectral_loss = environmental_loss(
                        delta, target_mel
                    )

                spectral_grad = tape.gradient(
                    spectral_loss, delta
                )

                spectral_optimizer.apply_gradients([
                    (spectral_grad, delta)
                ])

                delta.assign(
                    tf.clip_by_value(
                        delta, -epsilon, epsilon
                    )
                )

                repairs += 1

            total_repairs += repairs
            attack_losses.append(
                float(attack_loss.numpy())
            )

        waveform = delta.numpy()

        final_spectral = float(
            environmental_loss(
                delta, target_mel
            ).numpy()
        )

        rms = float(
            np.sqrt(np.mean(waveform ** 2))
        )

        history["attack_loss"].append(
            np.mean(attack_losses)
        )
        history["spectral_loss"].append(
            final_spectral
        )
        history["rms"].append(rms)
        history["repair_steps"].append(
            total_repairs
        )

        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(
                f"Epoch {epoch + 1:2d}/{epochs} | "
                f"Attack={np.mean(attack_losses):.4f} | "
                f"Spectral={final_spectral:.6f} | "
                f"RMS={rms:.6f} | "
                f"Repairs={total_repairs}"
            )

    return delta.numpy().copy(), history

In [ ]:
rain_constrained, constrained_history = (
    train_constrained_environmental_uap(
        initial_delta=rain_spectral_uap,
        target_mel=reference_mel_tf["rain"],
        tau=0.02,
        epsilon=0.003,
        epochs=30
    )
)

results = evaluate_perturbation(
    test_dataset,
    tf.constant(rain_constrained)
)

final_spectral = float(
    environmental_loss(
        tf.constant(rain_constrained),
        reference_mel_tf["rain"]
    ).numpy()
)

print("\nFinal results")
print("Accuracy:", results["accuracy"])
print("Attack loss:", results["loss"])
print("Spectral distance:", final_spectral)
print("Spectral constraint satisfied:", final_spectral <= 0.02)
print("Peak:", np.max(np.abs(rain_constrained)))
print("RMS:", np.sqrt(np.mean(rain_constrained ** 2)))

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    reference_mel["rain"],
    label="Reference rain",
    linewidth=2
)


uap = rain_constrained

distribution = uap_mel_distribution(
    tf.constant(uap)
).numpy()

plt.plot(
    distribution,
    label=f"UAP",
    alpha=0.8
)

plt.xlabel("Mel frequency band")
plt.ylabel("Normalized mel power")
plt.title("Rain Reference vs Learned UAP Spectra")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
thresholds = [0.05, 0.02, 0.01, 0.005, 0.001]

constrained_experiments = {}

for tau in thresholds:
    print(f"\n--- Rain | tau={tau} ---")

    uap, history = train_constrained_environmental_uap(
        initial_delta=rain_spectral_uap,
        target_mel=reference_mel_tf["rain"],
        tau=tau,
        epsilon=0.003,
        epochs=30
    )

    spectral = float(
        environmental_loss(
            tf.constant(uap),
            reference_mel_tf["rain"]
        ).numpy()
    )

    constrained_experiments[tau] = {
        "waveform": uap,
        "history": history,
        "spectral_loss": spectral,
        "constraint_satisfied": spectral <= tau,
        "rms": np.sqrt(np.mean(uap ** 2))
    }

    results = evaluate_perturbation(
        test_dataset,
        tf.constant(rain_constrained)
    )

    print(f"Spectral distance: {spectral:.6f}")
    print(f"Constraint satisfied: {spectral <= tau}")
    print("Accuracy:", results["accuracy"])

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    reference_mel["rain"],
    label="Reference rain",
    linewidth=2
)

for tau in thresholds:
    uap = constrained_experiments[tau]["waveform"]

    distribution = uap_mel_distribution(
        tf.constant(uap)
    ).numpy()

    plt.plot(
        distribution,
        label=f"UAP tau={tau}",
        alpha=0.8
    )

plt.xlabel("Mel frequency band")
plt.ylabel("Normalized mel power")
plt.title("Rain Reference vs Learned UAP Spectra")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from IPython.display import Audio, display

print("Unrestricted UAP")
display(Audio(
    rain_experiments[0.0]["waveform"],
    rate=SR,
    normalize=False
))

print("Constrained rain UAP")
display(Audio(
    rain_constrained,
    rate=SR,
    normalize=False
))

print("Constrained rain UAP tau=0.001")
display(Audio(
    constrained_experiments[0.001]['waveform7'],
    rate=SR,
    normalize=False
))

print("Actual rain")
display(Audio(
    filtered_reference_audio["rain"][:N_SAMPLES],
    rate=SR,
    normalize=False
))

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

signals = {
    "Actual rain": filtered_reference_audio["rain"][:N_SAMPLES],
    "Unrestricted UAP": rain_experiments[0.0]["waveform"],
    "Constrained UAP (tau=0.001)": constrained_experiments[0.001]["waveform"]
}

fig, axes = plt.subplots(
    len(signals),
    1,
    figsize=(12, 9),
    sharex=True
)

for ax, (name, audio) in zip(axes, signals.items()):
    stft = librosa.stft(
        audio,
        n_fft=1024,
        hop_length=128
    )

    magnitude = np.abs(stft)

    # Normalize each signal for comparing structure.
    db = librosa.amplitude_to_db(
        magnitude,
        ref=np.max
    )

    librosa.display.specshow(
        db,
        sr=SR,
        hop_length=128,
        x_axis="time",
        y_axis="linear",
        ax=ax,
        vmin=-80,
        vmax=0
    )

    ax.set_title(name)
    ax.set_ylim(0, 12000)

plt.tight_layout()
plt.show()

## Natural mask 2

In [ ]:
import numpy as np
import tensorflow as tf

SR = 44100
N_SAMPLES = 11025

rain_audio = np.asarray(
    filtered_reference_audio["rain"],
    dtype=np.float32
)

n = len(rain_audio)

rain_train = rain_audio[:int(0.70 * n)]
rain_val = rain_audio[int(0.70 * n):int(0.85 * n)]
rain_test = rain_audio[int(0.85 * n):]

def rms(audio):
    return float(
        np.sqrt(np.mean(np.square(audio)))
    )

rain_train_rms = rms(rain_train)

print("Train RMS:", rain_train_rms)
print("Validation RMS:", rms(rain_val))
print("Test RMS:", rms(rain_test))

In [ ]:
def sample_rain_batch(
    rain_tensor,
    batch_size,
    target_rms
):
    max_start = (
        tf.shape(rain_tensor)[0] - N_SAMPLES + 1
    )

    starts = tf.random.uniform(
        shape=(batch_size,),
        minval=0,
        maxval=max_start,
        dtype=tf.int32
    )

    indices = (
        starts[:, tf.newaxis]
        + tf.range(N_SAMPLES)[tf.newaxis, :]
    )

    segments = tf.gather(
        rain_tensor, indices
    )

    # One global scaling factor preserves the
    # natural volume variation between rain segments.
    scale = target_rms / rain_train_rms

    return segments * scale

In [ ]:
def evaluate_rain_mask(
    batches,
    rain_source,
    carrier_rms,
    residual=None,
    seed=42
):
    rng = np.random.default_rng(seed)
    residual_rng = np.random.default_rng(seed + 1)

    rain_tensor = tf.constant(
        rain_source, dtype=tf.float32
    )

    source_rms = rms(rain_source)
    scale = carrier_rms / source_rms

    correct = 0
    total = 0

    for audio_batch, label_batch in batches:
        audio_batch = tf.cast(
            audio_batch, tf.float32
        )

        batch_size = int(
            tf.shape(audio_batch)[0]
        )

        starts = rng.integers(
            0,
            len(rain_source) - N_SAMPLES + 1,
            size=batch_size
        )

        indices = (
            starts[:, None]
            + np.arange(N_SAMPLES)[None, :]
        )

        carrier = tf.gather(
            rain_tensor, indices
        ) * scale

        mixture = audio_batch + carrier

        if residual is not None:
            residual = np.asarray(
                residual, dtype=np.float32
            )

            residual_length = len(residual)

            offsets = residual_rng.integers(
                0,
                residual_length - N_SAMPLES + 1,
                size=batch_size
            )

            residual_indices = (
                offsets[:, None]
                + np.arange(N_SAMPLES)[None, :]
            )

            residual_windows = residual[residual_indices]

            mixture = (
                mixture
                + tf.constant(
                    residual_windows, dtype=tf.float32
                )
            )

        mixture = tf.clip_by_value(
            mixture, -1.0, 1.0
        )

        predictions = model(
            tf_log_mel(mixture),
            training=False
        )

        predicted_labels = tf.argmax(
            predictions,
            axis=-1,
            output_type=tf.int32
        )

        true_labels = tf.reshape(
            tf.cast(label_batch, tf.int32),
            [-1]
        )

        correct += int(
            tf.reduce_sum(
                tf.cast(
                    predicted_labels == true_labels,
                    tf.int32
                )
            ).numpy()
        )

        total += batch_size

    return correct / total

In [ ]:
RAIN_LEVELS = [
    0.003,
    0.005,
    0.007,
    0.010,
    0.015
]

pilot_batches = list(train_dataset.take(12))

for level in RAIN_LEVELS:
    accuracy = evaluate_rain_mask(
        pilot_batches,
        rain_val,
        carrier_rms=level,
        seed=42
    )

    print(
        f"Rain RMS={level:.4f} | "
        f"Accuracy={accuracy:.2%}"
    )

In [ ]:
def train_rain_residual(
    carrier_rms=0.003,
    residual_rms_ratio=0.40,
    residual_peak_ratio=0.80,
    epochs=80,
    learning_rate=1e-5,
    seed=42,
    random_phase=False
):
    tf.random.set_seed(seed)
    rng = np.random.default_rng(seed)

    rain_tensor = tf.constant(
        rain_train,
        dtype=tf.float32
    )

    max_rms = (
        carrier_rms * residual_rms_ratio
    )

    max_peak = (
        carrier_rms * residual_peak_ratio
    )

    # Small nonzero initialization.
    initial = rng.normal(
        0.0,
        1e-5,
        size=N_SAMPLES
    ).astype(np.float32)

    delta = tf.Variable(initial)

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    history = {
        "attack_loss": [],
        "residual_rms": [],
        "residual_peak": []
    }

    for epoch in range(epochs):
        losses = []

        for audio_batch, label_batch in train_dataset:
            audio_batch = tf.cast(
                audio_batch, tf.float32
            )

            batch_size = tf.shape(
                audio_batch
            )[0]

            carrier = sample_rain_batch(
                rain_tensor,
                batch_size,
                carrier_rms
            )

            with tf.GradientTape() as tape:
                if random_phase:
                    # Different circular shift for every
                    # keystroke in the batch.
                    shifts = tf.random.uniform(
                        shape=(batch_size,),
                        minval=0,
                        maxval=N_SAMPLES,
                        dtype=tf.int32
                    )

                    indices = (
                        tf.range(N_SAMPLES)[None, :]
                        + shifts[:, None]
                    ) % N_SAMPLES

                    residual_windows = tf.gather(
                        delta, indices
                    )

                else:
                    residual_windows = delta[None, :]

                mixture = tf.clip_by_value(
                    audio_batch + carrier + residual_windows,
                    -1.0,
                    1.0
                )

                predictions = model(
                    tf_log_mel(mixture),
                    training=False
                )

                attack_loss = loss_fn(
                    label_batch,
                    predictions
                )

            gradient = tape.gradient(
                attack_loss,
                delta
            )

            # Gradient ascent on attack loss.
            optimizer.apply_gradients([
                (-gradient, delta)
            ])

            # Peak amplitude constraint.
            delta.assign(
                tf.clip_by_value(
                    delta,
                    -max_peak,
                    max_peak
                )
            )

            # RMS constraint.
            current_rms = tf.sqrt(
                tf.reduce_mean(
                    tf.square(delta)
                ) + 1e-20
            )

            scale = tf.minimum(
                1.0,
                max_rms / current_rms
            )

            delta.assign(delta * scale)

            losses.append(
                float(attack_loss.numpy())
            )

        waveform = delta.numpy()

        mean_loss = float(
            np.mean(losses)
        )

        current_rms = rms(waveform)
        current_peak = float(
            np.max(np.abs(waveform))
        )

        history["attack_loss"].append(
            mean_loss
        )

        history["residual_rms"].append(
            current_rms
        )

        history["residual_peak"].append(
            current_peak
        )

        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(
                f"Epoch {epoch + 1:2d}/{epochs} | "
                f"Attack={mean_loss:.4f} | "
                f"Residual RMS={current_rms:.6f} | "
                f"Peak={current_peak:.6f}"
            )

    return delta.numpy().copy(), history

In [ ]:
CARRIER_RMS = 0.010

rain_residual, residual_history = (
    train_rain_residual(
        carrier_rms=CARRIER_RMS,
        residual_rms_ratio=0.05,
        residual_peak_ratio=0.10,
        epochs=30
    )
)

print("Final residual RMS:", rms(rain_residual))
print(
    "Final residual peak:",
    np.max(np.abs(rain_residual))
)

In [ ]:
rain_only_accuracy = evaluate_rain_mask(
    pilot_batches,
    rain_val,
    carrier_rms=CARRIER_RMS,
    seed=42
)

adversarial_accuracy = evaluate_rain_mask(
    pilot_batches,
    rain_val,
    carrier_rms=CARRIER_RMS,
    residual=rain_residual,
    seed=42
)

print("Rain-only accuracy:", rain_only_accuracy)
print("Rain + residual accuracy:", adversarial_accuracy)

In [ ]:
from IPython.display import Audio, display

DURATION = 5
length = SR * DURATION

# A continuous segment from held-out rain audio.
rain_segment = rain_val[:length].copy()

rain_segment *= (
    CARRIER_RMS / rms(rain_val)
)

# Repeat the learned 0.25-second residual.
repeats = int(
    np.ceil(length / len(rain_residual))
)

residual_long = np.tile(
    rain_residual, repeats
)[:length]

final_mask = np.clip(
    rain_segment + residual_long,
    -1.0,
    1.0
)

print("Rain alone")
display(Audio(
    rain_segment,
    rate=SR,
    normalize=False
))

print("Rain + adversarial residual")
display(Audio(
    final_mask,
    rate=SR,
    normalize=False
))

print("Residual alone (diagnostic)")
display(Audio(
    residual_long,
    rate=SR,
    normalize=False
))

In [ ]:
CARRIER_RMS = 0.0030
RATIOS = [0.025, 0.05, 0.10, 0.20, 0.40]

residual_sweep = {}

for ratio in RATIOS:
    print(f"\n--- Residual ratio: {ratio:.1%} ---")

    residual, history = train_rain_residual(
        carrier_rms=CARRIER_RMS,
        residual_rms_ratio=ratio,
        residual_peak_ratio=2 * ratio,
        epochs=30,
        learning_rate=1e-5,
        seed=42
    )

    accuracy = evaluate_rain_mask(
        pilot_batches,
        rain_val,
        carrier_rms=CARRIER_RMS,
        residual=residual,
        seed=42
    )

    residual_sweep[ratio] = {
        "waveform": residual,
        "history": history,
        "accuracy": accuracy,
        "rms": rms(residual),
        "peak": float(np.max(np.abs(residual)))
    }

    print(f"Accuracy: {accuracy:.2%}")
    print(f"Residual RMS: {rms(residual):.6f}")

In [ ]:
import pandas as pd

rain_only_accuracy = evaluate_rain_mask(
    pilot_batches,
    rain_val,
    carrier_rms=CARRIER_RMS,
    seed=42
)

rows = []

for ratio, experiment in residual_sweep.items():
    rows.append({
        "Ratio": ratio,
        "Accuracy": experiment["accuracy"],
        "Accuracy improvement (pp)": (
            rain_only_accuracy - experiment["accuracy"]
        ) * 100,
        "Residual RMS": experiment["rms"],
        "Residual peak": experiment["peak"]
    })

results_df = pd.DataFrame(rows)

print(f"Rain-only accuracy: {rain_only_accuracy:.2%}")
display(results_df)

In [ ]:
def make_random_residual_like(
    residual,
    seed=123
):
    rng = np.random.default_rng(seed)

    noise = rng.normal(
        size=residual.shape
    ).astype(np.float32)

    target_rms = rms(residual)

    noise *= target_rms / rms(noise)

    return noise


for ratio, experiment in residual_sweep.items():
    random_residual = make_random_residual_like(
        experiment["waveform"]
    )

    random_accuracy = evaluate_rain_mask(
        pilot_batches,
        rain_val,
        carrier_rms=CARRIER_RMS,
        residual=random_residual,
        seed=42
    )

    print(
        f"Ratio={ratio:.1%} | "
        f"Learned={experiment['accuracy']:.2%} | "
        f"Random={random_accuracy:.2%}"
    )

In [ ]:
residual_long = np.tile(
    residual_sweep[0.4]['waveform'], repeats
)[:length]

final_mask = np.clip(
    rain_segment + residual_long,
    -1.0,
    1.0
)

print("Rain + adversarial residual")
display(Audio(
    final_mask,
    rate=SR,
    normalize=False
))

print("adversarial residual")
display(Audio(
    residual_long,
    rate=SR,
    normalize=False
))

In [ ]:
def train_long_rain_residual(
    carrier_rms=0.003,
    residual_rms_ratio=0.40,
    residual_peak_ratio=0.80,
    duration=4.0,
    epochs=40,
    learning_rate=1e-5,
    seed=42
):
    tf.random.set_seed(seed)
    rng = np.random.default_rng(seed)

    residual_length = int(SR * duration)

    rain_tensor = tf.constant(
        rain_train, dtype=tf.float32
    )

    max_rms = carrier_rms * residual_rms_ratio
    max_peak = carrier_rms * residual_peak_ratio

    initial = rng.normal(
        0.0, 1e-5, residual_length
    ).astype(np.float32)

    delta = tf.Variable(initial)

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    history = {
        "attack_loss": [],
        "residual_rms": [],
        "residual_peak": []
    }

    for epoch in range(epochs):
        losses = []

        for audio_batch, label_batch in train_dataset:
            audio_batch = tf.cast(
                audio_batch, tf.float32
            )

            batch_size = tf.shape(audio_batch)[0]

            carrier = sample_rain_batch(
                rain_tensor,
                batch_size,
                carrier_rms
            )

            # Each example receives a random window
            # from the same longer residual.
            starts = tf.random.uniform(
                shape=(batch_size,),
                minval=0,
                maxval=residual_length - N_SAMPLES + 1,
                dtype=tf.int32
            )

            indices = (
                starts[:, tf.newaxis]
                + tf.range(N_SAMPLES)[tf.newaxis, :]
            )

            with tf.GradientTape() as tape:
                residual_windows = tf.gather(
                    delta, indices
                )

                mixture = tf.clip_by_value(
                    audio_batch
                    + carrier
                    + residual_windows,
                    -1.0,
                    1.0
                )

                predictions = model(
                    tf_log_mel(mixture),
                    training=False
                )

                attack_loss = loss_fn(
                    label_batch,
                    predictions
                )

            gradient = tape.gradient(
                attack_loss, delta
            )

            # Convert gather's potentially sparse gradient
            # into a dense gradient for the Adam update.
            gradient = tf.convert_to_tensor(gradient)

            optimizer.apply_gradients([
                (-gradient, delta)
            ])

            delta.assign(
                tf.clip_by_value(
                    delta, -max_peak, max_peak
                )
            )

            current_rms = tf.sqrt(
                tf.reduce_mean(tf.square(delta)) + 1e-20
            )

            delta.assign(
                delta * tf.minimum(
                    1.0, max_rms / current_rms
                )
            )

            losses.append(float(attack_loss.numpy()))

        waveform = delta.numpy()

        history["attack_loss"].append(
            float(np.mean(losses))
        )
        history["residual_rms"].append(
            rms(waveform)
        )
        history["residual_peak"].append(
            float(np.max(np.abs(waveform)))
        )

        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(
                f"Epoch {epoch+1:2d}/{epochs} | "
                f"Attack={np.mean(losses):.4f} | "
                f"RMS={rms(waveform):.6f} | "
                f"Peak={np.max(np.abs(waveform)):.6f}"
            )

    return delta.numpy().copy(), history

In [ ]:
long_residual, long_history = train_long_rain_residual(
    carrier_rms=0.003,
    residual_rms_ratio=0.40,
    residual_peak_ratio=0.80,
    duration=2.0,
    epochs=80
)

In [ ]:
long_accuracy = evaluate_rain_mask(
    pilot_batches,
    rain_val,
    carrier_rms=0.003,
    residual=long_residual,
    seed=42
)

print(f"Long residual accuracy: {long_accuracy:.2%}")
print(f"Long residual RMS: {rms(long_residual):.6f}")

In [ ]:
from IPython.display import Audio, display

length = SR * 8

rain_segment = rain_val[:length].copy()
rain_segment *= 0.003 / rms(rain_val)

short_residual = residual_sweep[0.40]["waveform"]

short_loop = np.tile(
    short_residual,
    int(np.ceil(length / len(short_residual)))
)[:length]

long_loop = np.tile(
    long_residual,
    int(np.ceil(length / len(long_residual)))
)[:length]

print("Original rain")
display(Audio(
    rain_segment, rate=SR, normalize=False
))

print("Rain + short repeating residual")
display(Audio(
    rain_segment + short_loop,
    rate=SR,
    normalize=False
))

print("Rain + longer residual")
display(Audio(
    rain_segment + long_loop,
    rate=SR,
    normalize=False
))

print("short residual")
display(Audio(
    short_loop,
    rate=SR,
    normalize=False
))

print("longer residual")
display(Audio(
    long_loop,
    rate=SR,
    normalize=False
))

In [ ]:
long_residual_4s, long_history_4s = train_long_rain_residual(
    carrier_rms=0.003,
    residual_rms_ratio=0.40,
    residual_peak_ratio=0.80,
    duration=4.0,
    epochs=80
)

In [ ]:
long_accuracy_4s = evaluate_rain_mask(
    pilot_batches,
    rain_val,
    carrier_rms=0.003,
    residual=long_residual_4s,
    seed=42
)

print(f"Long residual accuracy: {long_accuracy_4s:.2%}")
print(f"Long residual RMS: {rms(long_residual_4s):.6f}")

In [ ]:
import numpy as np

short_residual = np.asarray(
    residual_sweep[0.40]["waveform"],
    dtype=np.float32
)

# 0.25 seconds × 8 = 2 seconds
tiled_short = np.tile(short_residual, 8)

print("Short RMS:", rms(short_residual))
print("Tiled RMS:", rms(tiled_short))

In [ ]:
conditions = {
    "Rain only": None,
    "Short (fixed phase)": short_residual,
    "Short (random phase)": tiled_short,
    "Long 2s (random window)": long_residual,
    "Long 4s (random window)": long_residual_4s
}

for name, residual in conditions.items():
    accuracy = evaluate_rain_mask(
        pilot_batches,
        rain_val,
        carrier_rms=0.003,
        residual=residual,
        seed=42
    )

    print(f"{name}: {accuracy:.2%}")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf


def evaluate_paired_masks(
    batches,
    rain_source,
    carrier_rms,
    original_short,
    phase_short,
    long_residual,
    seed=42
):
    """
    Evaluate masking strategies under matched conditions.

    Conditions:
      1. Rain only
      2. Original short residual, fixed phase
      3. Original short residual, random phase
      4. Phase-trained short residual, random phase
      5. Long residual, random window

    For each example:
      - The same keystroke and rain crop are used.
      - Random-phase conditions use matching temporal offsets.
      - The rain source is scaled to the requested carrier RMS.

    Returns:
      Dictionary of accuracies and actual sampled carrier RMS.
    """

    rain_source = np.asarray(
        rain_source, dtype=np.float32
    ).reshape(-1)

    original_short = np.asarray(
        original_short, dtype=np.float32
    ).reshape(-1)

    phase_short = np.asarray(
        phase_short, dtype=np.float32
    ).reshape(-1)

    long_residual = np.asarray(
        long_residual, dtype=np.float32
    ).reshape(-1)

    # Validate dimensions.
    assert len(rain_source) >= N_SAMPLES
    assert len(original_short) == N_SAMPLES
    assert len(phase_short) == N_SAMPLES
    assert len(long_residual) > N_SAMPLES

    source_rms = rms(rain_source)
    assert source_rms > 0

    # Correct rain normalization.
    rain_scale = carrier_rms / source_rms

    # Independent random streams.
    rain_rng = np.random.default_rng(seed)
    offset_rng = np.random.default_rng(seed + 1)

    names = [
        "rain_only",
        "original_fixed",
        "original_random",
        "phase_random",
        "long_random"
    ]

    correct = {name: 0 for name in names}

    total = 0
    rain_energy = 0.0

    sample_indices = np.arange(N_SAMPLES)

    for audio_batch, label_batch in batches:

        audio_batch = tf.cast(
            audio_batch, tf.float32
        )

        labels = tf.reshape(
            tf.cast(label_batch, tf.int32),
            [-1]
        )

        batch_size = int(tf.shape(audio_batch)[0])

        # ----------------------------------
        # Sample identical rain for all conditions.
        # ----------------------------------

        rain_starts = rain_rng.integers(
            0,
            len(rain_source) - N_SAMPLES + 1,
            size=batch_size
        )

        rain_indices = (
            rain_starts[:, None]
            + sample_indices[None, :]
        )

        carrier_np = (
            rain_source[rain_indices] * rain_scale
        )

        carrier = tf.convert_to_tensor(
            carrier_np, dtype=tf.float32
        )

        rain_energy += np.sum(
            carrier_np.astype(np.float64) ** 2
        )

        # ----------------------------------
        # Sample long-residual windows.
        # ----------------------------------

        long_starts = offset_rng.integers(
            0,
            len(long_residual) - N_SAMPLES + 1,
            size=batch_size
        )

        long_indices = (
            long_starts[:, None]
            + sample_indices[None, :]
        )

        long_windows = long_residual[long_indices]

        # ----------------------------------
        # Use the same offsets modulo the
        # short waveform length.
        # ----------------------------------

        short_indices = (
            long_starts[:, None]
            + sample_indices[None, :]
        ) % N_SAMPLES

        original_random_windows = (
            original_short[short_indices]
        )

        phase_random_windows = (
            phase_short[short_indices]
        )

        # ----------------------------------
        # Construct all five conditions.
        # ----------------------------------

        base_mixture = audio_batch + carrier

        conditions = {
            "rain_only": None,

            "original_fixed": np.broadcast_to(
                original_short,
                (batch_size, N_SAMPLES)
            ),

            "original_random": original_random_windows,

            "phase_random": phase_random_windows,

            "long_random": long_windows
        }

        # ----------------------------------
        # Evaluate each condition.
        # ----------------------------------

        for name, residual in conditions.items():

            if residual is None:
                mixture = base_mixture
            else:
                mixture = (
                    base_mixture
                    + tf.convert_to_tensor(
                        residual, dtype=tf.float32
                    )
                )

            mixture = tf.clip_by_value(
                mixture, -1.0, 1.0
            )

            predictions = model(
                tf_log_mel(mixture),
                training=False
            )

            predicted_labels = tf.argmax(
                predictions,
                axis=-1,
                output_type=tf.int32
            )

            correct[name] += int(
                tf.reduce_sum(
                    tf.cast(
                        predicted_labels == labels,
                        tf.int32
                    )
                ).numpy()
            )

        total += batch_size

    results = {
        name: correct[name] / total
        for name in names
    }

    results["actual_carrier_rms"] = np.sqrt(
        rain_energy / (total * N_SAMPLES)
    )

    return results

In [ ]:
phase_robust_residual, phase_history = train_rain_residual(
    carrier_rms=0.003,
    residual_rms_ratio=0.40,
    residual_peak_ratio=0.80,
    epochs=80,
    learning_rate=1e-5,
    seed=42,
    random_phase=True
)

In [ ]:
phase_robust_tiled = np.tile(
    phase_robust_residual, 8
)

conditions = {
    "Rain only": None,
    "Original short, fixed": residual_sweep[0.40]["waveform"],
    "Original short, random": np.tile(
        residual_sweep[0.40]["waveform"], 8
    ),
    "Phase-trained short, random": phase_robust_tiled,
    "Long, random": long_residual
}

for name, residual in conditions.items():
    accuracy = evaluate_rain_mask(
        pilot_batches,
        rain_val,
        carrier_rms=0.003,
        residual=residual,
        seed=42
    )

    print(f"{name}: {accuracy:.2%}")

In [ ]:
from IPython.display import Audio, display

length = SR * 8

rain_segment = rain_val[:length].copy()
rain_segment *= 0.003 / rms(rain_val)

short_residual = phase_robust_residual

short_loop = np.tile(
    short_residual,
    int(np.ceil(length / len(short_residual)))
)[:length]

long_loop = np.tile(
    long_residual,
    int(np.ceil(length / len(long_residual)))
)[:length]

print("Original rain")
display(Audio(
    rain_segment, rate=SR, normalize=False
))

print("Rain + short repeating residual")
display(Audio(
    rain_segment + short_loop,
    rate=SR,
    normalize=False
))

print("Rain + longer residual")
display(Audio(
    rain_segment + long_loop,
    rate=SR,
    normalize=False
))

print("short residual")
display(Audio(
    short_loop,
    rate=SR,
    normalize=False
))

print("short residual")
display(Audio(
    short_residual,
    rate=SR,
    normalize=False
))

print("longer residual")
display(Audio(
    long_residual,
    rate=SR,
    normalize=False
))

print("longer residual")
display(Audio(
    long_residual_4s,
    rate=SR,
    normalize=False
))

In [ ]:
rows = []

for seed in range(100, 130):

    result = evaluate_paired_masks(
        batches=pilot_batches,
        rain_source=rain_val,
        carrier_rms=0.003,
        original_short=original_short,
        phase_short=phase_robust_residual,
        long_residual=long_residual,
        seed=seed
    )

    rows.append({
        "seed": seed,

        "rain_only": result["rain_only"],
        "original_fixed": result["original_fixed"],
        "original_random": result["original_random"],
        "phase_random": result["phase_random"],
        "long_random": result["long_random"],

        # Positive means long is more effective.
        "long_advantage_pp": 100 * (
            result["phase_random"]
            - result["long_random"]
        ),

        "actual_carrier_rms": (
            result["actual_carrier_rms"]
        )
    })

df = pd.DataFrame(rows)

display(
    df.drop(columns=["seed"]).agg(
        ["mean", "std", "min", "max"]
    )
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))

plt.plot(
    df["seed"],
    df["long_advantage_pp"],
    marker="o"
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel("Random seed")
plt.ylabel("Long residual advantage (pp)")
plt.title(
    "Paired comparison: long vs phase-trained short"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
test_batches = list(test_dataset)

test_rows = []

for seed in range(100, 130):
    result = evaluate_paired_masks(
        batches=test_batches,
        rain_source=rain_test,
        carrier_rms=0.003,
        original_short=residual_sweep[0.40]["waveform"],
        phase_short=phase_robust_residual,
        long_residual=long_residual,
        seed=seed
    )

    test_rows.append(result)

test_df = pd.DataFrame(test_rows)

display(test_df.agg(["mean", "std", "min", "max"]))

In [ ]:
CARRIER_RMS = 0.0050
RATIOS = [0.025, 0.05, 0.10, 0.20, 0.40]

residual_sweep = {}

for ratio in RATIOS:
    print(f"\n--- Residual ratio: {ratio:.1%} ---")

    residual, history = train_rain_residual(
        carrier_rms=CARRIER_RMS,
        residual_rms_ratio=ratio,
        residual_peak_ratio=2 * ratio,
        epochs=30,
        learning_rate=1e-5,
        seed=42,
        random_phase=True
    )


    accuracy = evaluate_rain_mask(
        pilot_batches,
        rain_val,
        carrier_rms=CARRIER_RMS,
        residual=residual,
        seed=42
    )

    residual_sweep[ratio] = {
        "waveform": residual,
        "history": history,
        "accuracy": accuracy,
        "rms": rms(residual),
        "peak": float(np.max(np.abs(residual)))
    }

    print(f"Accuracy: {accuracy:.2%}")
    print(f"Residual RMS: {rms(residual):.6f}")

In [ ]:
from IPython.display import Audio, display

length = SR * 8

rain_segment = rain_val[:length].copy()
rain_segment *= 0.005 / rms(rain_val)

short_residual = residual_sweep[0.2]['waveform']

short_loop = np.tile(
    short_residual,
    int(np.ceil(length / len(short_residual)))
)[:length]

print("Original rain")
display(Audio(
    rain_segment, rate=SR, normalize=False
))

print("Rain + short repeating residual")
display(Audio(
    rain_segment + short_loop,
    rate=SR,
    normalize=False
))

print("short residual")
display(Audio(
    short_loop,
    rate=SR,
    normalize=False
))

print("short residual")
display(Audio(
    short_residual,
    rate=SR,
    normalize=False
))


## Generalize

In [ ]:
print("Available audio sources:")

for name, audio in filtered_reference_audio.items():
    print(
        f"{name:15s} "
        f"Duration: {len(audio) / SR:.2f}s | "
        f"RMS: {rms(audio):.6f}"
    )

In [ ]:
import numpy as np

def split_carrier_audio(audio, train_frac=0.70, val_frac=0.15):
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    n = len(audio)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    splits = {
        "train": audio[:train_end],
        "val": audio[train_end:val_end],
        "test": audio[val_end:]
    }

    for split_name, segment in splits.items():
        if len(segment) < N_SAMPLES:
            raise ValueError(
                f"{split_name} split is shorter than "
                f"{N_SAMPLES} samples"
            )

    return splits


carrier_splits = {
    name: split_carrier_audio(audio)
    for name, audio in filtered_reference_audio.items()
}

carrier_val = {
    name: splits["val"]
    for name, splits in carrier_splits.items()
}

carrier_test = {
    name: splits["test"]
    for name, splits in carrier_splits.items()
}

In [ ]:
import numpy as np
import tensorflow as tf


def evaluate_carrier_residuals(
    batches,
    carrier_source,
    carrier_rms,
    residuals,
    seed=42
):
    """
    Evaluate multiple adversarial residuals against one carrier.

    Parameters
    ----------
    batches:
        Iterable of (audio_batch, label_batch).

    carrier_source:
        Mono environmental audio waveform.

    carrier_rms:
        Target RMS for the environmental audio.

    residuals:
        Dictionary mapping residual names to waveforms.

    seed:
        Controls random carrier crops and residual offsets.

    Returns
    -------
    Dictionary containing:
        - carrier_only accuracy
        - accuracy for each residual
        - actual carrier RMS
        - number of evaluated examples
    """

    carrier_source = np.asarray(
        carrier_source, dtype=np.float32
    ).reshape(-1)

    if len(carrier_source) < N_SAMPLES:
        raise ValueError("Carrier is shorter than one keystroke window")

    source_rms = rms(carrier_source)

    if source_rms <= 0:
        raise ValueError("Carrier has zero RMS")

    carrier_scale = carrier_rms / source_rms

    residuals = {
        name: np.asarray(waveform, dtype=np.float32).reshape(-1)
        for name, waveform in residuals.items()
    }

    for name, waveform in residuals.items():
        if len(waveform) < N_SAMPLES:
            raise ValueError(
                f"Residual '{name}' is too short"
            )

    carrier_rng = np.random.default_rng(seed)
    phase_rng = np.random.default_rng(seed + 1)

    sample_indices = np.arange(N_SAMPLES)

    correct = {
        "carrier_only": 0,
        **{name: 0 for name in residuals}
    }

    total = 0
    carrier_energy = 0.0

    for audio_batch, label_batch in batches:

        audio_batch = tf.cast(
            audio_batch, tf.float32
        )

        labels = tf.reshape(
            tf.cast(label_batch, tf.int32),
            [-1]
        )

        batch_size = int(tf.shape(audio_batch)[0])

        # -----------------------------------
        # Sample environmental audio windows.
        # -----------------------------------

        carrier_starts = carrier_rng.integers(
            0,
            len(carrier_source) - N_SAMPLES + 1,
            size=batch_size
        )

        carrier_indices = (
            carrier_starts[:, None]
            + sample_indices[None, :]
        )

        carrier_np = (
            carrier_source[carrier_indices]
            * carrier_scale
        )

        carrier = tf.convert_to_tensor(
            carrier_np,
            dtype=tf.float32
        )

        carrier_energy += np.sum(
            carrier_np.astype(np.float64) ** 2
        )

        base_mixture = audio_batch + carrier

        # -----------------------------------
        # Sample random temporal offsets.
        # -----------------------------------

        # Shared phase quantiles across conditions.
        # Short residuals use circular shifts.
        # Long residuals use random valid windows.
        phase_uniform = phase_rng.random(batch_size)

        conditions = {
            "carrier_only": None
        }

        for name, waveform in residuals.items():

            if len(waveform) == N_SAMPLES:

                shifts = np.floor(
                    phase_uniform * N_SAMPLES
                ).astype(np.int64)

                indices = (
                    shifts[:, None]
                    + sample_indices[None, :]
                ) % N_SAMPLES

            else:

                max_start = (
                    len(waveform) - N_SAMPLES
                )

                starts = np.floor(
                    phase_uniform * (max_start + 1)
                ).astype(np.int64)

                indices = (
                    starts[:, None]
                    + sample_indices[None, :]
                )

            conditions[name] = waveform[indices]

        # -----------------------------------
        # Evaluate all conditions.
        # -----------------------------------

        for name, residual_windows in conditions.items():

            if residual_windows is None:
                mixture = base_mixture

            else:
                mixture = (
                    base_mixture
                    + tf.convert_to_tensor(
                        residual_windows,
                        dtype=tf.float32
                    )
                )

            mixture = tf.clip_by_value(
                mixture, -1.0, 1.0
            )

            predictions = model(
                tf_log_mel(mixture),
                training=False
            )

            predicted_labels = tf.argmax(
                predictions,
                axis=-1,
                output_type=tf.int32
            )

            correct[name] += int(
                tf.reduce_sum(
                    tf.cast(
                        predicted_labels == labels,
                        tf.int32
                    )
                ).numpy()
            )

        total += batch_size

    if total == 0:
        raise ValueError("No evaluation examples were provided")

    results = {
        name: count / total
        for name, count in correct.items()
    }

    results["actual_carrier_rms"] = np.sqrt(
        carrier_energy / (total * N_SAMPLES)
    )

    results["n_examples"] = total

    return results

In [ ]:
import pandas as pd

residuals = {
    "phase_short": phase_robust_residual,
    "original_short": residual_sweep[0.40]["waveform"],
    "long_residual": long_residual
}

test_batches = list(test_dataset)
rows = []

for carrier_name, carrier_audio in carrier_val.items():

    for seed in range(100, 130):

        result = evaluate_carrier_residuals(
            batches=test_batches,
            carrier_source=carrier_audio,
            carrier_rms=0.003,
            residuals=residuals,
            seed=seed
        )

        rows.append({
            "carrier": carrier_name,
            "seed": seed,

            "carrier_only": result["carrier_only"],
            "phase_short": result["phase_short"],
            "original_short": result["original_short"],
            "long_residual": result["long_residual"],

            "improvement_pp": 100 * (
                result["carrier_only"]
                - result["phase_short"]
            ),

            "actual_carrier_rms": result["actual_carrier_rms"]
        })

transfer_results = pd.DataFrame(rows)

summary = (
    transfer_results
    .groupby("carrier")
    .agg(
        carrier_only_mean=("carrier_only", "mean"),
        carrier_only_std=("carrier_only", "std"),

        residual_mean=("phase_short", "mean"),
        residual_std=("phase_short", "std"),

        original_mean=('original_short', 'mean'),
        original_std=('original_short', 'std'),

        long_mean=('long_residual', 'mean'),
        long_std=('long_residual', 'std'),

        improvement_mean=("improvement_pp", "mean"),
        improvement_std=("improvement_pp", "std"),

        actual_rms=("actual_carrier_rms", "mean")
    )
)

display(summary)

In [ ]:
import numpy as np
from IPython.display import Audio, display

def make_mask_preview(
    carrier_audio,
    residual,
    carrier_rms=0.003,
    duration=8.0,
    seed=42
):
    carrier_audio = np.asarray(
        carrier_audio, dtype=np.float32
    ).reshape(-1)

    residual = np.asarray(
        residual, dtype=np.float32
    ).reshape(-1)

    requested_samples = int(duration * SR)

    # Use the available recording length if shorter.
    n_samples = min(requested_samples, len(carrier_audio))

    if n_samples == 0:
        raise ValueError("Carrier recording is empty")

    rng = np.random.default_rng(seed)

    # Random continuous segment.
    start = rng.integers(
        0,
        len(carrier_audio) - n_samples + 1
    )

    carrier = carrier_audio[start:start + n_samples].copy()

    # Normalize using the entire source RMS,
    # consistent with our evaluation procedure.
    carrier *= carrier_rms / rms(carrier_audio)

    # Repeat residual with a random starting phase.
    phase = rng.integers(0, len(residual))

    indices = (
        np.arange(n_samples) + phase
    ) % len(residual)

    residual_loop = residual[indices]

    mixture = np.clip(
        carrier + residual_loop,
        -1.0, 1.0
    )

    return carrier, mixture, residual_loop

In [ ]:
print("phase-trained residual:")
display(Audio(phase_robust_residual, rate=SR, normalize=False))

for carrier_name in ["rain", "fan", "wind"]:
    carrier, mixture, residual_loop = make_mask_preview(
        carrier_audio=carrier_val[carrier_name],
        residual=phase_robust_residual,
        carrier_rms=0.003,
        duration=8.0,
        seed=42
    )

    print(f"\n--- {carrier_name.upper()} ---")
    print(f"Carrier RMS: {rms(carrier):.6f}")
    print(f"Residual RMS: {rms(residual_loop):.6f}")
    print(f"Mixture RMS: {rms(mixture):.6f}")

    print("Carrier only:")
    display(Audio(carrier, rate=SR, normalize=False))

    print("Carrier + phase-trained residual:")
    display(Audio(mixture, rate=SR, normalize=False))


In [ ]:
import numpy as np
import pandas as pd

def analyze_carrier_loudness(
    carrier_audio,
    residual,
    target_rms=0.003
):
    carrier = np.asarray(carrier_audio, dtype=np.float32)

    carrier = carrier * (
        target_rms / rms(carrier)
    )

    residual_rms = rms(residual)

    # Non-overlapping 0.25-second windows
    n_windows = len(carrier) // N_SAMPLES

    windows = carrier[:n_windows * N_SAMPLES].reshape(
        n_windows, N_SAMPLES
    )

    window_rms = np.sqrt(
        np.mean(windows ** 2, axis=1)
    )

    return {
        "carrier_rms": rms(carrier),
        "residual_rms": residual_rms,
        "min_window_rms": np.min(window_rms),
        "median_window_rms": np.median(window_rms),
        "max_window_rms": np.max(window_rms),
        "median_residual_ratio": residual_rms / np.median(window_rms),
        "max_residual_ratio": residual_rms / np.min(window_rms)
    }


rows = []

for name, audio in carrier_val.items():
    result = analyze_carrier_loudness(
        audio,
        phase_robust_residual
    )

    rows.append({
        "carrier": name,
        **result
    })

display(pd.DataFrame(rows).set_index("carrier"))

In [ ]:
residual_scales = [0.25, 0.50, 0.75, 1.00]

for carrier_name in ["fan", "wind"]:

    print(f"\n===== {carrier_name.upper()} =====")

    for scale in residual_scales:

        scaled_residual = (
            phase_robust_residual * scale
        )

        carrier, mixture, _ = make_mask_preview(
            carrier_audio=carrier_val[carrier_name],
            residual=scaled_residual,
            carrier_rms=0.003,
            duration=5.0,
            seed=42
        )

        print(
            f"Residual scale: {scale:.2f} | "
            f"RMS: {rms(scaled_residual):.6f}"
        )

        display(
            Audio(
                mixture,
                rate=SR,
                normalize=False
            )
        )

In [ ]:
import numpy as np
import pandas as pd

phase_residual = np.asarray(
    phase_robust_residual,
    dtype=np.float32
).reshape(-1)

scaled_residual = phase_residual * 0.25

residuals = {
    "original_phase": phase_residual,
    "scaled_phase_025": scaled_residual
}

for name, residual in residuals.items():
    print(
        f"{name:20s} "
        f"RMS={rms(residual):.6f} | "
        f"Peak={np.max(np.abs(residual)):.6f}"
    )

In [ ]:
test_batches = list(test_dataset)

rows = []

for carrier_name, carrier_audio in carrier_val.items():

    for seed in range(100, 130):

        result = evaluate_carrier_residuals(
            batches=test_batches,
            carrier_source=carrier_audio,
            carrier_rms=0.003,
            residuals=residuals,
            seed=seed
        )

        rows.append({
            "carrier": carrier_name,
            "seed": seed,

            "carrier_only": result["carrier_only"],
            "original_phase": result["original_phase"],
            "scaled_phase_025": result["scaled_phase_025"],

            "original_improvement_pp": 100 * (
                result["carrier_only"]
                - result["original_phase"]
            ),

            "scaled_improvement_pp": 100 * (
                result["carrier_only"]
                - result["scaled_phase_025"]
            ),

            "actual_carrier_rms": result["actual_carrier_rms"]
        })

scaled_results = pd.DataFrame(rows)

scaled_summary = (
    scaled_results
    .groupby("carrier")
    .agg(
        carrier_only=("carrier_only", "mean"),
        original_accuracy=("original_phase", "mean"),
        scaled_accuracy=("scaled_phase_025", "mean"),
        original_improvement_pp=("original_improvement_pp", "mean"),
        scaled_improvement_pp=("scaled_improvement_pp", "mean"),
        scaled_improvement_std=("scaled_improvement_pp", "std")
    )
)

display(scaled_summary)

In [ ]:
import numpy as np
import tensorflow as tf


def train_carrier_residual(
    carrier_train,
    carrier_rms=0.003,
    residual_rms_ratio=0.10,
    residual_peak_ratio=0.20,
    epochs=80,
    learning_rate=1e-5,
    seed=42,
    batches=None
):
    """
    Train a universal adversarial residual against a frozen
    keystroke classifier using a specified environmental carrier.

    Assumes existing globals:
        model, tf_log_mel, loss_fn, train_dataset,
        SR, N_SAMPLES
    """

    if batches is None:
        batches = list(train_dataset.take(12))

    carrier_source = np.asarray(
        carrier_train, dtype=np.float32
    ).reshape(-1)

    if len(carrier_source) < N_SAMPLES:
        raise ValueError("Carrier recording is too short")

    source_rms = rms(carrier_source)

    if source_rms <= 0:
        raise ValueError("Carrier has zero RMS")

    carrier_scale = carrier_rms / source_rms

    rms_budget = carrier_rms * residual_rms_ratio
    peak_budget = carrier_rms * residual_peak_ratio

    rng = np.random.default_rng(seed)
    tf_rng = tf.random.Generator.from_seed(seed)

    # Start from zero; Adam will update the waveform.
    delta = tf.Variable(
        tf.zeros(N_SAMPLES, dtype=tf.float32)
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    sample_indices = np.arange(N_SAMPLES)

    history = []

    for epoch in range(epochs):

        epoch_losses = []

        for audio_batch, label_batch in batches:

            audio_batch = tf.cast(
                audio_batch, tf.float32
            )

            labels = tf.reshape(
                tf.cast(label_batch, tf.int32),
                [-1]
            )

            batch_size = int(tf.shape(audio_batch)[0])

            # Random carrier windows.
            starts = rng.integers(
                0,
                len(carrier_source) - N_SAMPLES + 1,
                size=batch_size
            )

            indices = (
                starts[:, None]
                + sample_indices[None, :]
            )

            carrier_windows = tf.convert_to_tensor(
                carrier_source[indices] * carrier_scale,
                dtype=tf.float32
            )

            # Random circular phase per keystroke.
            shifts = tf_rng.uniform(
                shape=(batch_size,),
                minval=0,
                maxval=N_SAMPLES,
                dtype=tf.int32
            )

            phase_indices = (
                tf.range(N_SAMPLES)[None, :]
                + shifts[:, None]
            ) % N_SAMPLES

            with tf.GradientTape() as tape:

                residual_windows = tf.gather(
                    delta, phase_indices
                )

                mixture = tf.clip_by_value(
                    audio_batch
                    + carrier_windows
                    + residual_windows,
                    -1.0,
                    1.0
                )

                predictions = model(
                    tf_log_mel(mixture),
                    training=False
                )

                loss = tf.reduce_mean(
                    loss_fn(labels, predictions)
                )

            gradients = tape.gradient(loss, delta)

            if gradients is None:
                raise RuntimeError(
                    "No gradient through the waveform preprocessing"
                )

            # Gradient ascent: maximize classifier loss.
            optimizer.apply_gradients([
                (-gradients, delta)
            ])

            # Project onto peak and RMS constraints.
            delta.assign(
                tf.clip_by_value(
                    delta,
                    -peak_budget,
                    peak_budget
                )
            )

            current_rms = tf.sqrt(
                tf.reduce_mean(tf.square(delta))
            )

            scale = tf.minimum(
                1.0,
                rms_budget / (current_rms + 1e-12)
            )

            delta.assign(delta * scale)

            epoch_losses.append(float(loss.numpy()))

        waveform = delta.numpy()

        history.append({
            "epoch": epoch + 1,
            "loss": np.mean(epoch_losses),
            "rms": rms(waveform),
            "peak": np.max(np.abs(waveform))
        })

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1:3d}/{epochs} | "
                f"Loss: {np.mean(epoch_losses):.4f} | "
                f"RMS: {rms(waveform):.6f} | "
                f"Peak: {np.max(np.abs(waveform)):.6f}"
            )

    return delta.numpy().copy(), pd.DataFrame(history)

In [ ]:
fan_residual, fan_history = train_carrier_residual(
    carrier_train=carrier_splits["fan"]["train"],
    carrier_rms=0.003,
    residual_rms_ratio=0.10,
    residual_peak_ratio=0.20,
    epochs=80,
    learning_rate=1e-5,
    seed=42
)

In [ ]:
wind_residual, wind_history = train_carrier_residual(
    carrier_train=carrier_splits["wind"]["train"],
    carrier_rms=0.003,
    residual_rms_ratio=0.10,
    residual_peak_ratio=0.20,
    epochs=80,
    learning_rate=1e-5,
    seed=42
)

In [ ]:
residuals = {
    "rain_scaled": scaled_residual,
    "fan_specific": fan_residual,
    "wind_specific": wind_residual
}

rows = []

for carrier_name, carrier_audio in carrier_val.items():

    for seed in range(100, 130):

        result = evaluate_carrier_residuals(
            batches=test_batches,
            carrier_source=carrier_audio,
            carrier_rms=0.003,
            residuals=residuals,
            seed=seed
        )

        rows.append({
            "carrier": carrier_name,
            "seed": seed,
            **result
        })

specific_results = pd.DataFrame(rows)

display(
    specific_results
    .groupby("carrier")[
        [
            "carrier_only",
            "rain_scaled",
            "fan_specific",
            "wind_specific"
        ]
    ]
    .agg(["mean", "std"])
)

In [ ]:
import numpy as np
import pandas as pd

carrier_rms_levels = [0.003, 0.005, 0.007, 0.010]

residuals = {
    "rain_scaled": scaled_residual,
    "fan_specific": fan_residual,
    "wind_specific": wind_residual
}

rows = []

for carrier_name in ["fan", "wind"]:

    carrier_audio = carrier_val[carrier_name]

    for target_rms in carrier_rms_levels:

        for seed in range(100, 130):

            result = evaluate_carrier_residuals(
                batches=test_batches,
                carrier_source=carrier_audio,
                carrier_rms=target_rms,
                residuals=residuals,
                seed=seed
            )

            for residual_name in residuals:

                rows.append({
                    "carrier": carrier_name,
                    "carrier_rms": target_rms,
                    "seed": seed,
                    "residual": residual_name,
                    "carrier_only": result["carrier_only"],
                    "masked_accuracy": result[residual_name],
                    "improvement_pp": 100 * (
                        result["carrier_only"]
                        - result[residual_name]
                    ),
                    "actual_carrier_rms": result["actual_carrier_rms"]
                })

rms_sweep_df = pd.DataFrame(rows)

rms_sweep_summary = (
    rms_sweep_df
    .groupby(["carrier", "carrier_rms", "residual"])
    .agg(
        carrier_only=("carrier_only", "mean"),
        masked_accuracy=("masked_accuracy", "mean"),
        improvement_pp=("improvement_pp", "mean"),
        improvement_std=("improvement_pp", "std")
    )
    .reset_index()
)

display(rms_sweep_summary)

In [ ]:
import matplotlib.pyplot as plt

for carrier_name in ["fan", "wind"]:

    subset = rms_sweep_summary[
        rms_sweep_summary["carrier"] == carrier_name
    ]

    plt.figure(figsize=(8, 5))

    baseline = subset[
        subset["residual"] == "rain_scaled"
    ]

    plt.plot(
        baseline["carrier_rms"],
        baseline["carrier_only"] * 100,
        "k--",
        label="Carrier only"
    )

    for residual_name in residuals:

        data = subset[
            subset["residual"] == residual_name
        ]

        plt.plot(
            data["carrier_rms"],
            data["masked_accuracy"] * 100,
            marker="o",
            label=residual_name
        )

    plt.title(f"{carrier_name.capitalize()} - Carrier RMS Sweep")
    plt.xlabel("Carrier RMS")
    plt.ylabel("Classifier Accuracy (%)")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
from IPython.display import Audio, display

for carrier_name in ["fan", "wind"]:

    print(f"\n===== {carrier_name.upper()} =====")

    for target_rms in [0.003, 0.005, 0.007, 0.010]:

        print(f"\nCarrier RMS: {target_rms}")

        carrier, _, _ = make_mask_preview(
            carrier_audio=carrier_val[carrier_name],
            residual=phase_robust_residual,
            carrier_rms=target_rms,
            duration=5.0,
            seed=42
        )

        print("Carrier only:")
        display(Audio(carrier, rate=SR, normalize=False))

        for scale in [0.25, 1.0]:

            residual = phase_robust_residual * scale

            _, mixture, _ = make_mask_preview(
                carrier_audio=carrier_val[carrier_name],
                residual=residual,
                carrier_rms=target_rms,
                duration=5.0,
                seed=42
            )

            print(
                f"Residual scale: {scale} | "
                f"Residual RMS: {rms(residual):.6f}"
            )

            display(
                Audio(
                    mixture,
                    rate=SR,
                    normalize=False
                )
            )

In [ ]:
import numpy as np
import pandas as pd

carrier_levels = [0.003, 0.005, 0.007, 0.010]
residual_scales = [0.25, 0.50, 0.75, 1.00]

residuals = {
    f"scale_{scale:.2f}": (
        np.asarray(phase_robust_residual, dtype=np.float32)
        * scale
    )
    for scale in residual_scales
}

rows = []

for carrier_name in ["fan", "wind"]:
    for carrier_rms in carrier_levels:
        for seed in range(100, 130):

            result = evaluate_carrier_residuals(
                batches=test_batches,
                carrier_source=carrier_val[carrier_name],
                carrier_rms=carrier_rms,
                residuals=residuals,
                seed=seed
            )

            for scale in residual_scales:
                key = f"scale_{scale:.2f}"

                rows.append({
                    "carrier": carrier_name,
                    "carrier_rms": carrier_rms,
                    "residual_scale": scale,
                    "residual_rms": rms(residuals[key]),
                    "seed": seed,
                    "carrier_only": result["carrier_only"],
                    "masked_accuracy": result[key],
                    "improvement_pp": 100 * (
                        result["carrier_only"] - result[key]
                    )
                })

scale_sweep_df = pd.DataFrame(rows)

scale_summary = (
    scale_sweep_df
    .groupby(["carrier", "carrier_rms", "residual_scale"])
    .agg(
        carrier_only=("carrier_only", "mean"),
        masked_accuracy=("masked_accuracy", "mean"),
        improvement_pp=("improvement_pp", "mean"),
        improvement_std=("improvement_pp", "std")
    )
    .reset_index()
)

display(scale_summary)

In [ ]:
import matplotlib.pyplot as plt

for carrier_name in ["fan", "wind"]:
    subset = scale_summary[
        scale_summary["carrier"] == carrier_name
    ]

    plt.figure(figsize=(8, 5))

    for carrier_rms in carrier_levels:
        data = subset[
            subset["carrier_rms"] == carrier_rms
        ].sort_values("residual_scale")

        plt.plot(
            data["residual_scale"],
            data["masked_accuracy"] * 100,
            marker="o",
            label=f"Carrier RMS {carrier_rms}"
        )

    plt.xlabel("Residual scale")
    plt.ylabel("Classifier accuracy (%)")
    plt.title(f"{carrier_name.capitalize()}: Residual Scale Sweep")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
fan_residual_010, fan_history_010 = train_carrier_residual(
    carrier_train=carrier_splits["fan"]["train"],
    carrier_rms=0.010,
    residual_rms_ratio=0.09,
    residual_peak_ratio=0.18,
    epochs=80,
    learning_rate=1e-5,
    seed=42
)

wind_residual_010, wind_history_010 = train_carrier_residual(
    carrier_train=carrier_splits["wind"]["train"],
    carrier_rms=0.010,
    residual_rms_ratio=0.12,
    residual_peak_ratio=0.24,
    epochs=80,
    learning_rate=1e-5,
    seed=42
)

In [ ]:
residuals = {
    "rain_scaled_025": phase_robust_residual * 0.25,
    "rain_scaled_075": phase_robust_residual * 0.75,
    "rain_full": phase_robust_residual,
    "fan_trained_010": fan_residual_010,
    "wind_trained_010": wind_residual_010
}

rows = []

for carrier_name in ["fan", "wind"]:
    for seed in range(100, 130):

        result = evaluate_carrier_residuals(
            batches=test_batches,
            carrier_source=carrier_val[carrier_name],
            carrier_rms=0.010,
            residuals=residuals,
            seed=seed
        )

        rows.append({
            "carrier": carrier_name,
            "seed": seed,
            **result
        })

comparison_df = pd.DataFrame(rows)

display(
    comparison_df.groupby("carrier")[
        ["carrier_only", *residuals.keys()]
    ].agg(["mean", "std"])
)

In [ ]:
import numpy as np
from IPython.display import Audio, display


def compare_residual_audio(
    carrier_name,
    residuals,
    carrier_rms=0.010,
    duration=5.0,
    seed=42,
    playback_gain=1.0
):
    """
    Listen to multiple residuals against an identical
    environmental carrier segment.

    residuals: dict[str, np.ndarray]
    """

    carrier_source = np.asarray(
        carrier_val[carrier_name],
        dtype=np.float32
    ).reshape(-1)

    n_samples = min(
        int(duration * SR),
        len(carrier_source)
    )

    if n_samples == 0:
        raise ValueError("Carrier recording is empty")

    rng = np.random.default_rng(seed)

    start = rng.integers(
        0,
        len(carrier_source) - n_samples + 1
    )

    # Match the normalization used in evaluation:
    # scale using the RMS of the entire source split.
    carrier = carrier_source[
        start:start + n_samples
    ].copy()

    carrier *= carrier_rms / rms(carrier_source)

    # Same phase starting point for all residuals.
    phase_fraction = rng.random()

    print(f"\n{'=' * 55}")
    print(f"{carrier_name.upper()} — Carrier RMS target: {carrier_rms}")
    print(f"{'=' * 55}")

    print(f"Carrier segment RMS: {rms(carrier):.6f}")

    print("\n[Carrier only]")
    display(
        Audio(
            carrier * playback_gain,
            rate=SR,
            normalize=False
        )
    )

    for name, residual in residuals.items():

        residual = np.asarray(
            residual,
            dtype=np.float32
        ).reshape(-1)

        phase = int(phase_fraction * len(residual))

        indices = (
            np.arange(n_samples) + phase
        ) % len(residual)

        residual_loop = residual[indices]

        mixture = np.clip(
            carrier + residual_loop,
            -1.0,
            1.0
        )

        print(f"\n[{name}]")
        print(
            f"Residual RMS: {rms(residual):.6f} | "
            f"Peak: {np.max(np.abs(residual)):.6f}"
        )
        print(
            f"Mixture RMS: {rms(mixture):.6f}"
        )

        display(
            Audio(
                mixture * playback_gain,
                rate=SR,
                normalize=False
            )
        )

In [ ]:
fan_listening_residuals = {
    "Rain-trained (0.75 scale)": phase_robust_residual * 0.75,
    "Fan-specific (RMS 0.010 training)": fan_residual_010,
    "Rain-trained (full scale)": phase_robust_residual
}

compare_residual_audio(
    carrier_name="fan",
    residuals=fan_listening_residuals,
    carrier_rms=0.010,
    duration=5.0,
    seed=42
)

In [ ]:
wind_listening_residuals = {
    "Rain-trained (full scale)": phase_robust_residual,
    "Wind-specific (RMS 0.010 training)": wind_residual_010,
    "Rain-trained (0.75 scale)": phase_robust_residual * 0.75
}

compare_residual_audio(
    carrier_name="wind",
    residuals=wind_listening_residuals,
    carrier_rms=0.010,
    duration=5.0,
    seed=42
)

In [ ]:
import numpy as np
import json
from pathlib import Path
from datetime import datetime, timezone

SAVE_DIR = Path("saved_residuals")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SR = 44100
N_SAMPLES = 11025

# Save trained waveforms at their native amplitudes.
residual_bank = {
    "rain_phase": phase_robust_residual,
    "fan_010": fan_residual_010,
    "wind_010": wind_residual_010,
}

# Include earlier source-specific experiments if available.
optional_residuals = {
    "fan_003": "fan_residual",
    "wind_003": "wind_residual",
}

for name, variable_name in optional_residuals.items():
    if variable_name in globals():
        residual_bank[name] = globals()[variable_name]

# Validate and convert.
residual_bank = {
    name: np.asarray(waveform, dtype=np.float32).reshape(-1)
    for name, waveform in residual_bank.items()
}

for name, waveform in residual_bank.items():
    assert len(waveform) == N_SAMPLES, (
        f"{name}: expected {N_SAMPLES} samples, "
        f"got {len(waveform)}"
    )
    assert np.all(np.isfinite(waveform)), (
        f"{name}: contains non-finite values"
    )

# Save all waveforms together.
np.savez_compressed(
    SAVE_DIR / "residual_bank.npz",
    **residual_bank
)

# Training and playback metadata.
training_info = {
    "rain_phase": {
        "training_carrier": "rain",
        "training_carrier_rms": 0.003,
        "phase_robust": True,
        "description": "Original phase-robust rain residual"
    },
    "fan_010": {
        "training_carrier": "fan",
        "training_carrier_rms": 0.010,
        "target_residual_rms": 0.0009,
        "phase_robust": True
    },
    "wind_010": {
        "training_carrier": "wind",
        "training_carrier_rms": 0.010,
        "target_residual_rms": 0.0012,
        "phase_robust": True
    },
    "fan_003": {
        "training_carrier": "fan",
        "training_carrier_rms": 0.003,
        "target_residual_rms": 0.0003,
        "phase_robust": True
    },
    "wind_003": {
        "training_carrier": "wind",
        "training_carrier_rms": 0.003,
        "target_residual_rms": 0.0003,
        "phase_robust": True
    }
}

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "sample_rate": SR,
    "n_samples": N_SAMPLES,
    "duration_seconds": N_SAMPLES / SR,
    "waveforms": {},
    "recommended_playback": {
        "rain": {
            "residual": "rain_phase",
            "carrier_rms": 0.003,
            "residual_scale": 1.0,
            "status": "Original rain configuration"
        },
        "fan": {
            "residual": "rain_phase",
            "carrier_rms": 0.010,
            "residual_scale": 0.75,
            "status": "Preferred in informal listening"
        },
        "wind": {
            "residual": "rain_phase",
            "carrier_rms": 0.010,
            "residual_scale": 1.0,
            "status": "Preferred for attack effectiveness; perceptually similar"
        }
    }
}

for name, waveform in residual_bank.items():
    manifest["waveforms"][name] = {
        **training_info.get(name, {}),
        "rms": float(rms(waveform)),
        "peak": float(np.max(np.abs(waveform))),
        "dtype": str(waveform.dtype)
    }

with open(SAVE_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved {len(residual_bank)} residuals to {SAVE_DIR}")

## Reloading Resids

In [ ]:
import numpy as np
import json
from pathlib import Path

SAVE_DIR = Path("saved_residuals")

with np.load(
    SAVE_DIR / "residual_bank.npz",
    allow_pickle=False
) as data:
    residual_bank = {
        name: data[name].copy()
        for name in data.files
    }

with open(SAVE_DIR / "manifest.json") as f:
    manifest = json.load(f)

print("Loaded residuals:")
for name, waveform in residual_bank.items():
    print(
        f"{name:15s} "
        f"RMS={rms(waveform):.6f} "
        f"Peak={np.max(np.abs(waveform)):.6f}"
    )

In [ ]:
fan_config = manifest["recommended_playback"]["fan"]
wind_config = manifest["recommended_playback"]["wind"]

fan_mask = (
    residual_bank[fan_config["residual"]]
    * fan_config["residual_scale"]
)

wind_mask = (
    residual_bank[wind_config["residual"]]
    * wind_config["residual_scale"]
)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

MULTI_CARRIER_CONFIG = {
    "rain": {
        "carrier_rms": 0.005,
        "residual_gain": 1.00
    },
    "fan": {
        "carrier_rms": 0.010,
        "residual_gain": 0.75
    },
    "wind": {
        "carrier_rms": 0.010,
        "residual_gain": 1.00
    }
}

carrier_train_sources = {
    name: carrier_splits[name]["train"]
    for name in MULTI_CARRIER_CONFIG
}

In [ ]:
def train_multi_carrier_residual(
    carrier_sources,
    carrier_config,
    train_dataset,
    initial_residual=None,
    base_rms_budget=0.0012,
    base_peak_budget=0.0024,
    epochs=80,
    steps_per_epoch=12,
    learning_rate=1e-5,
    seed=42
):
    """
    Train one phase-robust universal residual across
    multiple environmental audio carriers.

    carrier_sources:
        dict[name -> 1D training audio]

    carrier_config:
        dict[name -> {
            "carrier_rms": float,
            "residual_gain": float
        }]

    initial_residual:
        Optional existing waveform for warm-start training.

    Returns:
        residual, training_history
    """

    rng = np.random.default_rng(seed)

    carrier_names = list(carrier_config.keys())
    n_carriers = len(carrier_names)

    # -----------------------------------------
    # Prepare and normalize carrier recordings
    # -----------------------------------------

    prepared_carriers = {}

    for name in carrier_names:

        source = np.asarray(
            carrier_sources[name],
            dtype=np.float32
        ).reshape(-1)

        if len(source) < N_SAMPLES:
            raise ValueError(
                f"Carrier {name} is too short"
            )

        source_rms = rms(source)

        if source_rms <= 0:
            raise ValueError(
                f"Carrier {name} has zero RMS"
            )

        target_rms = carrier_config[name]["carrier_rms"]

        # Normalize by the full training split RMS.
        prepared_carriers[name] = (
            source * (target_rms / source_rms)
        )

    # -----------------------------------------
    # Initialize shared residual
    # -----------------------------------------

    if initial_residual is None:

        initial = np.zeros(
            N_SAMPLES,
            dtype=np.float32
        )

    else:

        initial = np.asarray(
            initial_residual,
            dtype=np.float32
        ).reshape(-1).copy()

        if len(initial) != N_SAMPLES:
            raise ValueError(
                "Initial residual has incorrect length"
            )

    delta = tf.Variable(
        initial,
        dtype=tf.float32
    )

    # -----------------------------------------
    # Projection function
    # -----------------------------------------

    def project_residual():

        # Enforce peak amplitude.
        delta.assign(
            tf.clip_by_value(
                delta,
                -base_peak_budget,
                base_peak_budget
            )
        )

        # Enforce RMS upper bound.
        current_rms = tf.sqrt(
            tf.reduce_mean(tf.square(delta))
        )

        scale = tf.minimum(
            1.0,
            base_rms_budget / (current_rms + 1e-12)
        )

        delta.assign(delta * scale)

    project_residual()

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    sample_indices = np.arange(N_SAMPLES)

    history = []

    # -----------------------------------------
    # Training loop
    # -----------------------------------------

    for epoch in range(epochs):

        epoch_losses = []
        carrier_loss_sums = {
            name: 0.0 for name in carrier_names
        }
        carrier_loss_counts = {
            name: 0 for name in carrier_names
        }

        for audio_batch, label_batch in train_dataset.take(
            steps_per_epoch
        ):

            audio_batch = tf.cast(
                audio_batch,
                tf.float32
            )

            labels = tf.reshape(
                tf.cast(label_batch, tf.int32),
                [-1]
            )

            batch_size = int(
                tf.shape(audio_batch)[0]
            )

            # ---------------------------------
            # Assign carrier types per example
            # ---------------------------------

            # Approximately equal representation
            # of all carriers in each batch.
            carrier_ids = np.tile(
                np.arange(n_carriers),
                int(np.ceil(batch_size / n_carriers))
            )[:batch_size]

            rng.shuffle(carrier_ids)

            carrier_windows = np.empty(
                (batch_size, N_SAMPLES),
                dtype=np.float32
            )

            residual_gains = np.empty(
                batch_size,
                dtype=np.float32
            )

            # ---------------------------------
            # Sample carrier windows
            # ---------------------------------

            for carrier_id, name in enumerate(carrier_names):

                positions = np.where(
                    carrier_ids == carrier_id
                )[0]

                if len(positions) == 0:
                    continue

                source = prepared_carriers[name]

                starts = rng.integers(
                    0,
                    len(source) - N_SAMPLES + 1,
                    size=len(positions)
                )

                indices = (
                    starts[:, None]
                    + sample_indices[None, :]
                )

                carrier_windows[positions] = source[indices]

                residual_gains[positions] = (
                    carrier_config[name]["residual_gain"]
                )

            carrier_tensor = tf.convert_to_tensor(
                carrier_windows,
                dtype=tf.float32
            )

            gain_tensor = tf.convert_to_tensor(
                residual_gains[:, None],
                dtype=tf.float32
            )

            # ---------------------------------
            # Random phase per example
            # ---------------------------------

            shifts = rng.integers(
                0,
                N_SAMPLES,
                size=batch_size
            )

            phase_indices = (
                sample_indices[None, :]
                + shifts[:, None]
            ) % N_SAMPLES

            phase_indices = tf.convert_to_tensor(
                phase_indices,
                dtype=tf.int32
            )

            # ---------------------------------
            # Forward pass
            # ---------------------------------

            with tf.GradientTape() as tape:

                residual_windows = tf.gather(
                    delta,
                    phase_indices
                )

                residual_windows *= gain_tensor

                mixture = tf.clip_by_value(
                    audio_batch
                    + carrier_tensor
                    + residual_windows,
                    -1.0,
                    1.0
                )

                predictions = model(
                    tf_log_mel(mixture),
                    training=False
                )

                # Existing classifier loss.
                loss = tf.reduce_mean(
                    loss_fn(labels, predictions)
                )

            # ---------------------------------
            # Gradient ascent
            # ---------------------------------

            gradients = tape.gradient(
                loss,
                delta
            )

            if gradients is None:
                raise RuntimeError(
                    "No gradient through preprocessing"
                )

            optimizer.apply_gradients([
                (-gradients, delta)
            ])

            project_residual()

            epoch_losses.append(
                float(loss.numpy())
            )

        # -------------------------------------
        # Epoch diagnostics
        # -------------------------------------

        waveform = delta.numpy()

        row = {
            "epoch": epoch + 1,
            "loss": float(np.mean(epoch_losses)),
            "rms": rms(waveform),
            "peak": float(
                np.max(np.abs(waveform))
            )
        }

        history.append(row)

        if (epoch + 1) % 10 == 0:

            print(
                f"Epoch {epoch + 1:3d}/{epochs} | "
                f"Loss: {row['loss']:.4f} | "
                f"RMS: {row['rms']:.6f} | "
                f"Peak: {row['peak']:.6f}"
            )

    return (
        delta.numpy().copy(),
        pd.DataFrame(history)
    )

In [ ]:
multi_residual_warm, multi_history_warm = (
    train_multi_carrier_residual(
        carrier_sources=carrier_train_sources,
        carrier_config=MULTI_CARRIER_CONFIG,
        train_dataset=train_dataset,
        initial_residual=phase_robust_residual,
        base_rms_budget=0.0012,
        base_peak_budget=0.0024,
        epochs=80,
        steps_per_epoch=12,
        learning_rate=1e-5,
        seed=42
    )
)

In [ ]:
multi_residual_scratch, multi_history_scratch = (
    train_multi_carrier_residual(
        carrier_sources=carrier_train_sources,
        carrier_config=MULTI_CARRIER_CONFIG,
        train_dataset=train_dataset,
        initial_residual=None,
        base_rms_budget=0.0012,
        base_peak_budget=0.0024,
        epochs=32 0,
        steps_per_epoch=12,
        learning_rate=1e-5,
        seed=42
    )
)

In [ ]:
rows = []

for carrier_name, config in MULTI_CARRIER_CONFIG.items():

    carrier_rms = config["carrier_rms"]
    gain = config["residual_gain"]

    residuals = {
        "rain_baseline": (
            phase_robust_residual * gain
        ),
        "multi_warm": (
            multi_residual_warm * gain
        ),
        "multi_scratch": (
            multi_residual_scratch * gain
        )
    }

    for seed in range(100, 130):

        result = evaluate_carrier_residuals(
            batches=test_batches,
            carrier_source=carrier_val[carrier_name],
            carrier_rms=carrier_rms,
            residuals=residuals,
            seed=seed
        )

        rows.append({
            "carrier": carrier_name,
            "seed": seed,
            "carrier_rms": carrier_rms,
            "residual_gain": gain,
            **result
        })

multi_results = pd.DataFrame(rows)

multi_summary = (
    multi_results
    .groupby("carrier")[
        [
            "carrier_only",
            "rain_baseline",
            "multi_warm",
            "multi_scratch"
        ]
    ]
    .agg(["mean", "std"])
)

display(multi_summary)

In [ ]:
residual_bank["multi_warm"] = multi_residual_warm
residual_bank["multi_scratch"] = multi_residual_scratch

for name, waveform in [
    ("multi_warm", multi_residual_warm),
    ("multi_scratch", multi_residual_scratch)
]:
    manifest["waveforms"][name] = {
        "training_carriers": list(MULTI_CARRIER_CONFIG),
        "carrier_config": MULTI_CARRIER_CONFIG,
        "initialization": (
            "rain_phase" if name == "multi_warm"
            else "zeros"
        ),
        "phase_robust": True,
        "epochs": 80,
        "learning_rate": 1e-5,
        "steps_per_epoch": 12,
        "seed": 42,
        "rms": float(rms(waveform)),
        "peak": float(np.max(np.abs(waveform)))
    }

np.savez_compressed(
    SAVE_DIR / "residual_bank.npz",
    **residual_bank
)

with open(SAVE_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

multi_history_warm.to_csv(
    SAVE_DIR / "multi_warm_history.csv",
    index=False
)

multi_history_scratch.to_csv(
    SAVE_DIR / "multi_scratch_history.csv",
    index=False
)

In [ ]:
residual_bank["multi_scratch_320"] = np.asarray(
    multi_residual_scratch,
    dtype=np.float32
).copy()

manifest["waveforms"]["multi_scratch_320"] = {
    "training_carriers": list(MULTI_CARRIER_CONFIG),
    "carrier_config": MULTI_CARRIER_CONFIG,
    "initialization": "zeros",
    "epochs": 320,
    "convergence_observed_around_epoch": 250,
    "phase_robust": True,
    "rms": float(rms(multi_residual_scratch)),
    "peak": float(np.max(np.abs(multi_residual_scratch)))
}

np.savez_compressed(
    SAVE_DIR / "residual_bank.npz",
    **residual_bank
)

with open(SAVE_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

multi_history_scratch.to_csv(
    SAVE_DIR / "multi_scratch_320_history.csv",
    index=False
)

In [ ]:
multi_listening_residuals = {
    "Rain baseline": phase_robust_residual,
    "Multi warm-start": multi_residual_warm,
    "Multi scratch": multi_residual_scratch
}

for carrier_name, config in MULTI_CARRIER_CONFIG.items():

    gain = config["residual_gain"]

    residuals = {
        name: waveform * gain
        for name, waveform in multi_listening_residuals.items()
    }

    compare_residual_audio(
        carrier_name=carrier_name,
        residuals=residuals,
        carrier_rms=config["carrier_rms"],
        duration=5.0,
        seed=42
    )

In [ ]:
residual_stats = []

for name, waveform in multi_listening_residuals.items():

    waveform = np.asarray(waveform, dtype=np.float32)

    residual_stats.append({
        "residual": name,
        "rms": rms(waveform),
        "peak": np.max(np.abs(waveform)),
        "peak_to_rms": np.max(np.abs(waveform)) / rms(waveform)
    })

display(pd.DataFrame(residual_stats))

In [ ]:
rows = []

for carrier_name, config in MULTI_CARRIER_CONFIG.items():

    gain = config["residual_gain"]

    residuals = {
        "rain_original": phase_robust_residual * gain,
        "warm_original": multi_residual_warm * gain,
        "scratch_original": multi_residual_scratch * gain,
        "warm_90": multi_residual_warm * gain * 0.90,
        "scratch_90": multi_residual_scratch * gain * 0.90,
    }

    for seed in range(100, 130):

        result = evaluate_carrier_residuals(
            batches=test_batches,
            carrier_source=carrier_val[carrier_name],
            carrier_rms=config["carrier_rms"],
            residuals=residuals,
            seed=seed
        )

        rows.append({
            "carrier": carrier_name,
            "seed": seed,
            **result
        })

amplitude_df = pd.DataFrame(rows)

display(
    amplitude_df.groupby("carrier")[
        [
            "carrier_only",
            "rain_original",
            "warm_original",
            "scratch_original",
            "warm_90",
            "scratch_90"
        ]
    ].agg(["mean", "std"])
)

## Comparisons

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

uap_path = Path("uap_epsilon_001.npy")

original_uap = np.load(
    uap_path,
    allow_pickle=False
).astype(np.float32).reshape(-1)

assert len(original_uap) == N_SAMPLES, (
    f"Expected {N_SAMPLES} samples, got {len(original_uap)}"
)
assert np.all(np.isfinite(original_uap))

def waveform_stats(name, waveform):
    waveform = np.asarray(waveform, dtype=np.float32)
    wave_rms = rms(waveform)
    peak = float(np.max(np.abs(waveform)))

    return {
        "name": name,
        "rms": wave_rms,
        "peak": peak,
        "peak_to_rms": peak / max(wave_rms, 1e-12)
    }

display(pd.DataFrame([
    waveform_stats("Original UAP", original_uap),
    waveform_stats("Rain baseline", phase_robust_residual),
    waveform_stats("Multi warm", multi_residual_warm),
    waveform_stats("Multi scratch", multi_residual_scratch)
]))

In [ ]:
BASE_RMS = 0.0012
BASE_PEAK = 0.0024

def scale_to_budget(waveform, target_rms, max_peak):
    """
    Uniformly scale a waveform to the largest amplitude
    satisfying both the RMS and peak constraints.

    Preserves the original waveform shape.
    """
    waveform = np.asarray(
        waveform, dtype=np.float32
    ).reshape(-1)

    current_rms = rms(waveform)
    current_peak = float(np.max(np.abs(waveform)))

    if current_rms <= 0 or current_peak <= 0:
        raise ValueError("Waveform must have nonzero amplitude")

    scale = min(
        target_rms / current_rms,
        max_peak / current_peak
    )

    return (waveform * scale).astype(np.float32)


original_uap_matched = scale_to_budget(
    original_uap,
    target_rms=BASE_RMS,
    max_peak=BASE_PEAK
)

display(pd.DataFrame([
    waveform_stats("Original UAP scaled", original_uap_matched),
    waveform_stats("Rain baseline", phase_robust_residual),
    waveform_stats("Multi scratch", multi_residual_scratch)
]))

In [ ]:
comparison_rows = []

for carrier_name, config in MULTI_CARRIER_CONFIG.items():

    carrier_rms = config["carrier_rms"]
    gain = config["residual_gain"] * 0.90

    residuals = {
        "original_uap": original_uap_matched * gain,
        "rain_baseline": phase_robust_residual * gain,
        "multi_warm": multi_residual_warm * gain,
        "multi_scratch": multi_residual_scratch * gain
    }

    for seed in range(100, 130):

        result = evaluate_carrier_residuals(
            batches=test_batches,
            carrier_source=carrier_val[carrier_name],
            carrier_rms=carrier_rms,
            residuals=residuals,
            seed=seed
        )

        comparison_rows.append({
            "carrier": carrier_name,
            "seed": seed,
            "carrier_rms": carrier_rms,
            **result
        })

uap_comparison_df = pd.DataFrame(comparison_rows)

uap_summary = (
    uap_comparison_df
    .groupby("carrier")[
        [
            "carrier_only",
            "original_uap",
            "rain_baseline",
            "multi_warm",
            "multi_scratch"
        ]
    ]
    .agg(["mean", "std"])
)

display(uap_summary)

In [ ]:
# Silent-carrier configuration.
# We reuse the rain recording as a dummy source,
# but its target RMS is zero, so it contributes
# no environmental audio to the mixture.

KEYSTROKE_ONLY_CONFIG = {
    "silence": {
        "carrier_rms": 0.0,
        "residual_gain": 1.0
    }
}

keystroke_only_uap, keystroke_uap_history = (
    train_multi_carrier_residual(
        carrier_sources={
            "silence": carrier_splits["rain"]["train"]
        },
        carrier_config=KEYSTROKE_ONLY_CONFIG,
        train_dataset=train_dataset,
        initial_residual=None,
        base_rms_budget=0.0012,
        base_peak_budget=0.0024,
        epochs=320,
        steps_per_epoch=12,
        learning_rate=1e-5,
        seed=42
    )
)

In [ ]:
comparison_residuals = {
    "original_uap": original_uap_matched,
    "keystroke_only_new": keystroke_only_uap,
    "rain_baseline": phase_robust_residual,
    "multi_warm": multi_residual_warm,
    "multi_scratch": multi_residual_scratch
}

rows = []

for carrier_name, config in MULTI_CARRIER_CONFIG.items():

    carrier_rms = config["carrier_rms"]
    gain = config["residual_gain"] * 0.90

    residuals = {
        name: waveform * gain
        for name, waveform in comparison_residuals.items()
    }

    for seed in range(100, 130):

        result = evaluate_carrier_residuals(
            batches=test_batches,
            carrier_source=carrier_val[carrier_name],
            carrier_rms=carrier_rms,
            residuals=residuals,
            seed=seed
        )

        rows.append({
            "carrier": carrier_name,
            "seed": seed,
            "carrier_rms": carrier_rms,
            **result
        })

final_uap_df = pd.DataFrame(rows)

conditions = [
    "carrier_only",
    "original_uap",
    "keystroke_only_new",
    "rain_baseline",
    "multi_warm",
    "multi_scratch"
]

display(
    final_uap_df.groupby("carrier")[conditions]
    .agg(["mean", "std"])
)

In [ ]:
residual_bank["keystroke_only_phase_320"] = (
    keystroke_only_uap.copy()
)

manifest["waveforms"]["keystroke_only_phase_320"] = {
    "training_carriers": [],
    "initialization": "zeros",
    "phase_robust": True,
    "epochs": 320,
    "steps_per_epoch": 12,
    "learning_rate": 1e-5,
    "seed": 42,
    "rms": float(rms(keystroke_only_uap)),
    "peak": float(np.max(np.abs(keystroke_only_uap)))
}

np.savez_compressed(
    SAVE_DIR / "residual_bank.npz",
    **residual_bank
)

with open(SAVE_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

keystroke_uap_history.to_csv(
    SAVE_DIR / "keystroke_only_phase_320_history.csv",
    index=False
)

In [ ]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf

EVAL_SEED = 20261116
N_REPEATS = 30
PLAYBACK_SCALE = 0.90

BASE_RESIDUAL_RMS = 0.0012
BASE_RESIDUAL_PEAK = 0.0024

# Make global randomness explicit as well.
random.seed(EVAL_SEED)
np.random.seed(EVAL_SEED)
tf.keras.utils.set_random_seed(EVAL_SEED)

# Optional for stricter TensorFlow reproducibility.
# May raise errors for unsupported nondeterministic operations.
tf.config.experimental.enable_op_determinism()


def make_random_residual(rng):
    """
    Non-adversarial waveform with exactly the target RMS,
    subject to the same peak upper bound.

    Uniform noise is used to avoid clipping artifacts.
    """
    noise = rng.uniform(
        -1.0, 1.0, size=N_SAMPLES
    ).astype(np.float32)

    noise *= BASE_RESIDUAL_RMS / rms(noise)

    assert rms(noise) <= BASE_RESIDUAL_RMS + 1e-7
    assert np.max(np.abs(noise)) <= BASE_RESIDUAL_PEAK

    return noise


def phase_shift_batch(waveform, phases):
    """
    Circularly shift one waveform by a potentially
    different phase for every example in the batch.
    """
    waveform = np.asarray(waveform, dtype=np.float32)

    indices = (
        np.arange(N_SAMPLES)[None, :]
        - phases[:, None]
    ) % N_SAMPLES

    return waveform[indices]


def sample_carrier_batch(source, starts, scale):
    """
    Sample carrier windows using predetermined offsets.
    The scale is based on the full carrier source RMS.
    """
    indices = (
        starts[:, None]
        + np.arange(N_SAMPLES)[None, :]
    )

    return source[indices] * scale


def paired_evaluate_trial(
    batches,
    carrier_source,
    carrier_rms,
    residuals,
    seed
):
    """
    Every condition receives identical:
      - keystroke recordings
      - carrier windows
      - residual phase offsets

    Returns accuracy and number of correct predictions.
    """
    rng = np.random.default_rng(seed)

    source = np.asarray(
        carrier_source, dtype=np.float32
    ).reshape(-1)

    assert len(source) >= N_SAMPLES

    source_rms = rms(source)
    assert source_rms > 0

    carrier_scale = carrier_rms / source_rms

    correct = {
        "carrier_only": 0,
        **{name: 0 for name in residuals}
    }

    total = 0

    for audio, labels in batches:

        audio = np.asarray(audio, dtype=np.float32)

        # Handle (batch, samples, 1) if present.
        if audio.ndim == 3 and audio.shape[-1] == 1:
            audio = audio[..., 0]

        assert audio.ndim == 2
        assert audio.shape[1] == N_SAMPLES

        labels = np.asarray(labels).reshape(-1)

        batch_size = len(labels)
        assert batch_size == len(audio)

        # Shared carrier windows.
        starts = rng.integers(
            0,
            len(source) - N_SAMPLES + 1,
            size=batch_size
        )

        # Shared residual phase offsets.
        phases = rng.integers(
            0,
            N_SAMPLES,
            size=batch_size
        )

        carrier = sample_carrier_batch(
            source, starts, carrier_scale
        )

        base_mixture = audio + carrier

        # Carrier-only baseline.
        predictions = model(
            tf_log_mel(
                tf.convert_to_tensor(base_mixture)
            ),
            training=False
        )

        predicted_labels = tf.argmax(
            predictions, axis=-1
        ).numpy()

        correct["carrier_only"] += int(
            np.sum(predicted_labels == labels)
        )

        # All residuals share the same carrier and phases.
        for name, residual in residuals.items():

            shifted = phase_shift_batch(
                residual, phases
            )

            mixture = base_mixture + shifted

            predictions = model(
                tf_log_mel(
                    tf.convert_to_tensor(mixture)
                ),
                training=False
            )

            predicted_labels = tf.argmax(
                predictions, axis=-1
            ).numpy()

            correct[name] += int(
                np.sum(predicted_labels == labels)
            )

        total += batch_size

    return {
        name: {
            "accuracy": count / total,
            "correct": count,
            "total": total
        }
        for name, count in correct.items()
    }

In [ ]:
# 1. Load the original UAP and scale it to budget
original_uap = np.load("uap_epsilon_001.npy", allow_pickle=False).astype(np.float32).reshape(-1)

def scale_to_budget(waveform, target_rms=0.0012, max_peak=0.0024):
    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)
    current_rms = rms(waveform)
    current_peak = float(np.max(np.abs(waveform)))
    scale = min(target_rms / current_rms, max_peak / current_peak)
    return (waveform * scale).astype(np.float32)

original_uap_matched = scale_to_budget(original_uap)

# 2. Recover the remaining waveforms from residual_bank
keystroke_only_uap = residual_bank["keystroke_only_phase_320"]
phase_robust_residual = residual_bank["rain_phase"]
multi_residual_warm = residual_bank["multi_warm"]
multi_residual_scratch = residual_bank["multi_scratch"]

print("Successfully recovered:")
print("  original_uap_matched (RMS: {:.6f})".format(rms(original_uap_matched)))
print("  keystroke_only_uap   (RMS: {:.6f})".format(rms(keystroke_only_uap)))
print("  phase_robust_residual(RMS: {:.6f})".format(rms(phase_robust_residual)))
print("  multi_residual_warm  (RMS: {:.6f})".format(rms(multi_residual_warm)))
print("  multi_residual_scratch (RMS: {:.6f})".format(rms(multi_residual_scratch)))

In [ ]:
test_batches = list(test_dataset)

learned_residuals = {
    "original_uap": original_uap_matched,
    "keystroke_only": keystroke_only_uap,
    "rain_baseline": phase_robust_residual,
    "multi_warm": multi_residual_warm,
    "multi_scratch": multi_residual_scratch,
}

# Verify all waveforms have the expected shape.
for name, waveform in learned_residuals.items():
    assert np.asarray(waveform).shape == (N_SAMPLES,), name

paired_rows = []

for carrier_idx, (carrier_name, config) in enumerate(
    MULTI_CARRIER_CONFIG.items()
):
    gain = (
        config["residual_gain"]
        * PLAYBACK_SCALE
    )

    for repeat in range(N_REPEATS):

        # Deterministic seed for this carrier and repeat.
        seed_sequence = np.random.SeedSequence([
            EVAL_SEED,
            carrier_idx,
            repeat
        ])

        trial_seed, noise_seed = [
            int(s) for s in seed_sequence.generate_state(2)
        ]

        noise_rng = np.random.default_rng(noise_seed)

        # Generate a new random waveform for each repeat.
        random_residual = make_random_residual(noise_rng)

        residuals = {
            "random_noise": random_residual * gain,
            **{
                name: np.asarray(waveform, dtype=np.float32) * gain
                for name, waveform in learned_residuals.items()
            }
        }

        results = paired_evaluate_trial(
            batches=test_batches,
            carrier_source=carrier_val[carrier_name],
            carrier_rms=config["carrier_rms"],
            residuals=residuals,
            seed=trial_seed
        )

        paired_rows.append({
            "carrier": carrier_name,
            "repeat": repeat,
            "seed": trial_seed,
            **{
                name: result["accuracy"]
                for name, result in results.items()
            }
        })

paired_df = pd.DataFrame(paired_rows)

conditions = [
    "carrier_only",
    "random_noise",
    "original_uap",
    "keystroke_only",
    "rain_baseline",
    "multi_warm",
    "multi_scratch"
]

paired_summary = (
    paired_df.groupby("carrier")[conditions]
    .agg(["mean", "std"])
)

display(paired_summary)

In [ ]:
paired_differences = paired_df[
    ["carrier", "repeat"]
].copy()

for name in conditions[1:]:
    paired_differences[name] = (
        paired_df["carrier_only"] - paired_df[name]
    ) * 100

# Positive means multi-scratch reduces accuracy more
# than the keystroke-only baseline.
paired_differences["multi_vs_keystroke"] = (
    paired_df["keystroke_only"]
    - paired_df["multi_scratch"]
) * 100

display(
    paired_differences
    .groupby("carrier")
    .agg({
        name: ["mean", "std"]
        for name in [
            "random_noise",
            "original_uap",
            "keystroke_only",
            "rain_baseline",
            "multi_warm",
            "multi_scratch",
            "multi_vs_keystroke"
        ]
    })
)

In [ ]:
paired_df.to_csv(
    SAVE_DIR / "paired_evaluation_90.csv",
    index=False
)

paired_summary.to_csv(
    SAVE_DIR / "paired_evaluation_90_summary.csv"
)

paired_differences.to_csv(
    SAVE_DIR / "paired_evaluation_90_differences.csv",
    index=False
)

In [ ]:
def collect_paired_predictions(
    batches,
    carrier_source,
    carrier_rms,
    residuals,
    seed,
    carrier_name,
    repeat
):
    """
    Collect per-example predictions for all conditions.

    All conditions share:
      - the same keystroke examples
      - the same carrier windows
      - the same residual phase offsets

    Returns one row per example per condition.
    """
    rng = np.random.default_rng(seed)

    source = np.asarray(
        carrier_source, dtype=np.float32
    ).reshape(-1)

    assert len(source) >= N_SAMPLES
    assert rms(source) > 0

    carrier_scale = carrier_rms / rms(source)

    rows = []
    sample_offset = 0

    def predict(mixture):
        features = tf_log_mel(
            tf.convert_to_tensor(
                mixture, dtype=tf.float32
            )
        )

        outputs = model(features, training=False)

        return tf.argmax(
            outputs, axis=-1
        ).numpy().reshape(-1)

    for audio, labels in batches:

        audio = np.asarray(audio, dtype=np.float32)

        if audio.ndim == 3 and audio.shape[-1] == 1:
            audio = audio[..., 0]

        labels = np.asarray(labels).reshape(-1).astype(int)

        assert audio.ndim == 2
        assert audio.shape[1] == N_SAMPLES
        assert len(audio) == len(labels)

        batch_size = len(labels)

        starts = rng.integers(
            0,
            len(source) - N_SAMPLES + 1,
            size=batch_size
        )

        phases = rng.integers(
            0,
            N_SAMPLES,
            size=batch_size
        )

        carrier = sample_carrier_batch(
            source, starts, carrier_scale
        )

        base_mixture = audio + carrier

        conditions = {
            "carrier_only": base_mixture
        }

        for name, residual in residuals.items():
            shifted = phase_shift_batch(
                residual, phases
            )
            conditions[name] = base_mixture + shifted

        for condition_name, mixture in conditions.items():

            predictions = predict(mixture)

            for i in range(batch_size):
                rows.append({
                    "carrier": carrier_name,
                    "repeat": repeat,
                    "sample_id": sample_offset + i,
                    "condition": condition_name,
                    "true_label": int(labels[i]),
                    "predicted_label": int(predictions[i]),
                    "correct": bool(
                        predictions[i] == labels[i]
                    ),
                    "carrier_start": int(starts[i]),
                    "residual_phase": int(phases[i])
                })

        sample_offset += batch_size

    return pd.DataFrame(rows)

In [ ]:
PREDICTION_REPEATS = 30

learned_residuals = {
    "original_uap": original_uap_matched,
    "keystroke_only": keystroke_only_uap,
    "rain_baseline": phase_robust_residual,
    "multi_warm": multi_residual_warm,
    "multi_scratch": multi_residual_scratch,
}

prediction_frames = []

for carrier_idx, (carrier_name, config) in enumerate(
    MULTI_CARRIER_CONFIG.items()
):
    gain = (
        config["residual_gain"]
        * PLAYBACK_SCALE
    )

    for repeat in range(PREDICTION_REPEATS):

        seed_sequence = np.random.SeedSequence([
            EVAL_SEED,
            carrier_idx,
            repeat
        ])

        trial_seed, noise_seed = [
            int(s)
            for s in seed_sequence.generate_state(2)
        ]

        noise_rng = np.random.default_rng(noise_seed)
        random_residual = make_random_residual(noise_rng)

        residuals = {
            "random_noise": random_residual * gain,
            **{
                name: np.asarray(
                    waveform, dtype=np.float32
                ) * gain
                for name, waveform in learned_residuals.items()
            }
        }

        trial_predictions = collect_paired_predictions(
            batches=test_batches,
            carrier_source=carrier_val[carrier_name],
            carrier_rms=config["carrier_rms"],
            residuals=residuals,
            seed=trial_seed,
            carrier_name=carrier_name,
            repeat=repeat
        )

        prediction_frames.append(trial_predictions)

prediction_df = pd.concat(
    prediction_frames,
    ignore_index=True
)

display(prediction_df.head())

In [ ]:
prediction_accuracy = (
    prediction_df
    .groupby(["carrier", "condition"])["correct"]
    .mean()
    .unstack()
)

display(prediction_accuracy)

if "paired_df" in globals():
    for carrier_name in MULTI_CARRIER_CONFIG:
        for condition in conditions:
            previous = paired_df.loc[
                paired_df["carrier"] == carrier_name,
                condition
            ].mean()

            current = prediction_accuracy.loc[
                carrier_name, condition
            ]

            assert np.isclose(
                previous, current, atol=1e-6
            ), (carrier_name, condition, previous, current)

    print("Prediction accuracies match paired evaluation.")

In [ ]:
# Supply the exact class order used during model training.
# Leave as None to display numeric class IDs.
CLASS_NAMES = CLASSES

def class_name(class_id, class_names=CLASS_NAMES):
    class_id = int(class_id)

    if class_names is None:
        return str(class_id)

    return str(class_names[class_id])


def common_predictions(
    df,
    carrier,
    condition,
    top_n=10,
    errors_only=False,
    class_names=None
):
    subset = df.loc[
        (df["carrier"] == carrier)
        & (df["condition"] == condition)
    ]

    if errors_only:
        subset = subset.loc[~subset["correct"]]

    counts = (
        subset["predicted_label"]
        .value_counts()
        .head(top_n)
    )

    total = len(subset)

    return pd.DataFrame({
        "class": [
            class_name(i, class_names)
            for i in counts.index
        ],
        "count": counts.values,
        "share": (
            counts.values / total
            if total else np.zeros(len(counts))
        )
    })


# Example: top incorrect predictions for multi-scratch on fan
display(
    common_predictions(
        prediction_df,
        carrier="fan",
        condition="multi_scratch",
        top_n=10,
        errors_only=True,
        class_names=CLASS_NAMES
    )
)

In [ ]:
def compare_prediction_distributions(
    df,
    carrier,
    errors_only=True,
    top_n=5,
    class_names=None
):
    rows = []

    for condition in conditions:

        top = common_predictions(
            df,
            carrier=carrier,
            condition=condition,
            top_n=top_n,
            errors_only=errors_only,
            class_names=class_names
        )

        for rank, (_, item) in enumerate(
            top.iterrows(), start=1
        ):
            rows.append({
                "condition": condition,
                "rank": rank,
                "predicted_class": item["class"],
                "share": item["share"]
            })

    return pd.DataFrame(rows)


display(
    compare_prediction_distributions(
        prediction_df,
        carrier="fan",
        errors_only=True,
        class_names=CLASS_NAMES
    )
)

In [ ]:
def prediction_concentration(
    df,
    class_names=None
):
    rows = []

    for (carrier, condition), group in df.groupby(
        ["carrier", "condition"]
    ):
        all_counts = group["predicted_label"].value_counts()

        errors = group.loc[~group["correct"]]
        error_counts = errors["predicted_label"].value_counts()

        top_class = int(all_counts.index[0])

        if len(error_counts):
            top_error_class = int(error_counts.index[0])
            top_error_share = (
                error_counts.iloc[0] / len(errors)
            )
        else:
            top_error_class = None
            top_error_share = np.nan

        rows.append({
            "carrier": carrier,
            "condition": condition,
            "accuracy": group["correct"].mean(),
            "top_predicted_class": class_name(
                top_class, class_names
            ),
            "top_class_share": (
                all_counts.iloc[0] / len(group)
            ),
            "top_error_class": (
                class_name(top_error_class, class_names)
                if top_error_class is not None else None
            ),
            "top_error_share": top_error_share,
            "unique_predicted_classes": len(all_counts)
        })

    return pd.DataFrame(rows)


concentration_df = prediction_concentration(
    prediction_df,
    class_names=CLASS_NAMES
)

display(concentration_df)

In [ ]:
import matplotlib.pyplot as plt

def plot_prediction_heatmap(
    df,
    carrier,
    top_n=15,
    errors_only=False,
    class_names=None
):
    subset = df[df["carrier"] == carrier].copy()

    if errors_only:
        subset = subset[~subset["correct"]]

    # Select the most common classes across all conditions.
    top_classes = (
        subset["predicted_label"]
        .value_counts()
        .head(top_n)
        .index
    )

    counts = pd.crosstab(
        subset["condition"],
        subset["predicted_label"]
    )

    counts = counts.reindex(
        index=conditions,
        columns=top_classes,
        fill_value=0
    )

    # Normalize by all predictions/errors for each condition,
    # not just the displayed top classes.
    totals = subset.groupby("condition").size()
    percentages = (
        counts.div(totals, axis=0).fillna(0) * 100
    )

    fig, ax = plt.subplots(figsize=(12, 5))

    im = ax.imshow(
        percentages.to_numpy(),
        aspect="auto",
        cmap="Blues",
        vmin=0
    )

    ax.set_xticks(range(len(top_classes)))
    ax.set_xticklabels(
        [class_name(i, class_names) for i in top_classes],
        rotation=45,
        ha="right"
    )

    ax.set_yticks(range(len(conditions)))
    ax.set_yticklabels(conditions)

    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Condition")

    mode = "Incorrect predictions" if errors_only else "All predictions"
    ax.set_title(f"{carrier.capitalize()} — {mode}")

    fig.colorbar(
        im,
        ax=ax,
        label="Share of predictions (%)"
    )

    plt.tight_layout()
    plt.show()


for carrier_name in MULTI_CARRIER_CONFIG:
    plot_prediction_heatmap(
        prediction_df,
        carrier=carrier_name,
        top_n=15,
        errors_only=True,
        class_names=CLASS_NAMES
    )

In [ ]:
prediction_df.to_csv(
    SAVE_DIR / "paired_predictions_90.csv",
    index=False
)

concentration_df.to_csv(
    SAVE_DIR / "prediction_concentration_90.csv",
    index=False
)

In [ ]:
for i, name in enumerate(CLASSES):
    print(f"{i} -> {name}")

In [ ]:
def analyze_target_class(df, target_class=1):
    rows = []

    for (carrier, condition), group in df.groupby(
        ["carrier", "condition"]
    ):
        other_classes = group[
            group["true_label"] != target_class
        ]

        target_examples = group[
            group["true_label"] == target_class
        ]

        false_positive_rate = (
            (other_classes["predicted_label"] == target_class).mean()
        )

        target_recall = (
            (target_examples["predicted_label"] == target_class).mean()
            if len(target_examples) else np.nan
        )

        rows.append({
            "carrier": carrier,
            "condition": condition,
            "accuracy": group["correct"].mean(),
            "target_false_positive_rate": false_positive_rate,
            "target_recall": target_recall,
            "target_prediction_share": (
                group["predicted_label"] == target_class
            ).mean(),
            "true_target_share": (
                group["true_label"] == target_class
            ).mean()
        })

    return pd.DataFrame(rows)


target_analysis = analyze_target_class(
    prediction_df,
    target_class=1
)

display(
    target_analysis.sort_values(
        ["carrier", "target_false_positive_rate"],
        ascending=[True, False]
    )
)